**MATERIAL DOCENTE — NO DISTRIBUIR**

*Solución del LAB S02. No se entrega al alumno.*

# LAB S02: Ingesta, lectura, limpieza y normalización de documentos

**Objetivo del módulo (temario):** implementar procesos de **ingesta, lectura, limpieza y normalización** de documentos de diferentes formatos.

**Objetivo del LAB:** al terminar, el participante abre PDF, DOCX, correos, HTML e imágenes con la biblioteca adecuada, normaliza el texto extraído, separa los archivos defectuosos en una cuarentena sin detener el lote y reúne todo en una clase de carga, `CargadorDocumentos`, que procesa un lote de 20 archivos mixtos y entrega sus estadísticas. Las funciones pasan al paquete `pipeline_sbs` (clonado desde GitHub) y, en el caso, se encadenan en un primer pipeline, con un commit por etapa.

**En palabras simples:** este LAB enseña a cargar una carpeta de archivos de una entidad (oficios, estados financieros, contratos, reclamos), a sacar el texto de cada uno y a separar los que no se pueden leer, sin que un archivo dañado detenga el trabajo.

**Mapa del cuaderno** (cada encabezado indica el verbo del objetivo que trabaja y su ítem del temario):

| Bloque | Minutos | Verbo del objetivo | Ítem |
|---|---|---|---|
| Antes del reto: repaso y verificación (C00, C0, C0b, C0c) | medio, no objetivo del módulo | | |
| Reto de apertura | 6 | Ingesta | M2.1 |
| Ejercicio 1: un lector por formato | 15 | Lectura | M2.2, M2.4 |
| Ejercicio 2: tablas con tabula-py y Camelot | 8 | Lectura | M2.4 |
| Ejercicio 3: funciones de normalización | 10 | Limpieza y normalización | M2.3 |
| Ejercicio 4: errores, cuarentena, duplicados y fichas | 10 | Ingesta | M2.1, M2.4 |
| Ejercicio 5: la clase de carga | 16 | Integra todo | M2.5 |
| Caso, paso 1: primer pipeline básico | 12 a 15 de los 50 del caso | Integra todo | M2.1 a M2.5 |
| Cierre del cuaderno | | Verificación del objetivo | |

**Cómo trabajar**
1. Las celdas se ejecutan en orden (`Shift + Enter`).
2. El participante completa las celdas marcadas con `# TODO`.
3. Cada ejercicio termina con un **checkpoint** (`assert`, una comprobación automática) y con una celda **Guardar** (≈1 min, dentro del tiempo del ejercicio). Si un checkpoint falla, el mensaje indica qué revisar y no significa que el participante se equivocó.
4. Tiempo estimado: 65 min de práctica en CPU (reto 6, E1 15, E2 8, E3 10, E4 10, E5 16), más el paso 1 del caso.

> ⚠ **El LAB trabaja solo con documentos ficticios.** No se usan documentos internos de la institución del participante ni datos personales reales, y no se suben a servicios de IA en la nube.

> Este cuaderno se abre desde la carpeta `05_Laboratorios` del repositorio clonado; se guarda con `Ctrl+S` antes de cada commit.

## Antes del reto (repaso y verificación): los dos repositorios, el verificador y el primer `push`

Esta sección repasa y verifica lo hecho en los bloques previos (introducción a GitHub, creación de la cuenta y repositorio del docente) y guarda el repositorio del docente en el GitHub del participante. **Todo se hace desde este cuaderno**, con celdas: no hace falta abrir otra ventana. Los bloques de GitHub son un medio para guardar el trabajo; el objetivo del módulo empieza en el reto.

**Dos repositorios, dos usos (no se mezclan)**

| | `origin`: repositorio del docente | `mio`: repositorio del participante |
|---|---|---|
| Dirección | `https://github.com/jonatanfigueroagil-creator/entorno-extraccion-documental.git` (público) | `https://github.com/<su-usuario>/extraccion-documental-sbs.git` (privado y vacío) |
| Para qué sirve | traer los programas: solo se clona y se hace `git pull` | guardar lo que el participante hace en cada ejercicio |
| Quién escribe | solo el docente | solo el participante |

**Insumo:** el repositorio ya clonado en `C:\curso\trabajo\entorno-extraccion-documental` y el verificador del entorno de la raíz del repositorio.

**Tarea:** (1) ejecutar la celda que conecta el paquete (más abajo); (2) ejecutar el verificador (control C00); (3) en la sección «Su repositorio», guardar el repositorio del docente en el GitHub propio con tres celdas; (4) ejecutar los controles C0, C0b y C0c.

**En palabras simples:** `origin` es la fuente de los programas del docente y `mio` es la carpeta personal del participante en GitHub. `commit` guarda una versión en el historial de la VM y `push` la copia a GitHub: mientras no se hace el `push`, nada está en GitHub.

**Los cuatro controles de esta sección**
- **C00:** el verificador del entorno termina en 0 errores (celda de más abajo).
- **C0:** `pytest` pasa sobre el paquete clonado.
- **C0b:** hay al menos un commit propio, es decir, que no está en `origin/main`.
- **C0c:** el remoto `mio` existe y la rama `trabajo-s02` lo sigue; la confirmación final se hace en el navegador: en github.com, dentro de `extraccion-documental-sbs`, la rama `trabajo-s02` es visible.

**Si falla:**
- `git clone` no conecta con github.com: el respaldo es descomprimir `06_PIPELINE_INICIO_S02.zip` en `C:\curso\paquete`; sin historial Git, los controles C0b y C0c avisan y no bloquean el LAB.
- Sin cuenta o sin acceso a github.com: el participante sigue con commits locales, avisa al docente y retoma la subida después.
- Para traer cambios nuevos del docente se usa `git pull origin main`.

La verificación (C00, C0, C0b y C0c) está más abajo, después de la celda que conecta el paquete.

## 0. Localización de los datos de práctica

Esta celda solo busca dónde están los documentos de práctica (la carpeta `02_Datos_Practica`), como cuando se ubica la carpeta del expediente antes de revisarlo. No cambia nada. **Debe verse:** `Datos de práctica encontrados:` y la lista de subcarpetas, entre ellas `01_Lote_S02`. **Si falla:** `FileNotFoundError` significa solo que el cuaderno se movió de carpeta; se abre de nuevo desde la carpeta de la sesión. No se escriben rutas absolutas: si el notebook se mueve fuera del paquete del curso, no encontrará los datos.

In [7]:
# Celda estándar de localización de datos (no se modifica)
import os
from pathlib import Path


def localizar_carpeta(nombre="02_Datos_Practica", max_niveles=6, obligatorio=True):
    """Busca `nombre` en la carpeta actual y en sus carpetas superiores."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents][: max_niveles + 1]:
        candidata = carpeta / nombre
        if candidata.is_dir():
            return candidata
    # Respaldo: carpetas del curso en la unidad del sistema (VM de laboratorio)
    curso = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso"
    respaldo = [curso / "paquete" / nombre]
    if nombre == "02_Datos_Practica":
        respaldo.insert(0, curso / "datos")
    for candidata in respaldo:
        if candidata.is_dir():
            return candidata
    if obligatorio:
        raise FileNotFoundError(
            f"No se encuentra la carpeta '{nombre}'. El notebook se abre desde su carpeta "
            "dentro del paquete del curso, sin moverlo de lugar."
        )
    return None


def localizar_datos():
    return localizar_carpeta("02_Datos_Practica")


DATOS = localizar_datos()
print("Datos de práctica encontrados:", DATOS.name)
print("Subcarpetas:", sorted(p.name for p in DATOS.iterdir() if p.is_dir()))

Datos de práctica encontrados: 02_Datos_Practica
Subcarpetas: ['01_Lote_S02', '02_Oficios', '03_Estados_Financieros', '04_Contratos', '05_Polizas', '06_Reclamos', '07_Tarifarios', '08_Vouchers', '09_Formularios', '10_Ground_Truth', '11_Salidas_Precalculadas', '12_Actas', '13_Expedientes']


## Conexión del paquete `pipeline_sbs`

Los ejercicios **importan** del paquete que está junto a este cuaderno, en la carpeta `04_pipeline_sbs` del repositorio clonado en el bloque «Repositorio del docente» (`C:\curso\trabajo\entorno-extraccion-documental\04_pipeline_sbs`) o, si no hubo Internet, del respaldo descomprimido de `06_PIPELINE_INICIO_S02.zip` en `C:\curso\paquete`. Esta celda lo deja listo (no se modifica). S02 corre en CPU. Las exportaciones y las pruebas usan siempre el mismo paquete: el clon o, de respaldo, una copia de trabajo temporal. En palabras simples: `pipeline_sbs` es la caja de herramientas del proyecto del equipo y esta celda la deja abierta sobre la mesa. **Debe verse:** una línea con las versiones de los programas. **Si falla:** `ModuleNotFoundError` significa que falta clonar el repositorio (bloque «Repositorio del docente») o instalarlo con `pip install -e`; se avisa al docente.

In [8]:
# Librerías del LAB y paquete pipeline_sbs (no se modifica)
import hashlib
import json
import os
import re
import shutil
import sys
import tempfile
import time
import unicodedata
import zipfile
from dataclasses import asdict, dataclass, field
from datetime import date, datetime
from decimal import Decimal
from email import policy
from email.parser import BytesParser

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
import pandas as pd
import pdfplumber
import pymupdf
import pypdf
from bs4 import BeautifulSoup
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph
from PIL import Image

pymupdf.TOOLS.mupdf_display_errors(False)


_PAQUETE_TRABAJO = None      # paquete único de trabajo: el mismo para importar, exportar y probar


def _copia_de_trabajo(origen):
    """Copia el paquete `origen` a una carpeta temporal para no escribir sobre un paquete ajeno o de solo lectura."""
    destino = Path(tempfile.mkdtemp(prefix="pipeline_sbs_trabajo_")) / "04_pipeline_sbs"
    shutil.copytree(origen, destino, ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache", "*.egg-info"))
    print("Se usa una copia de trabajo temporal del paquete (no se encontró el repositorio clonado).")
    return destino


def localizar_paquete():
    """Raíz del paquete de trabajo (carpeta con pipeline_sbs/): el clon de GitHub, el respaldo en C:\\curso\\paquete o,
    si no hay ninguno, una copia temporal. Siempre devuelve el mismo paquete durante la sesión."""
    global _PAQUETE_TRABAJO
    if _PAQUETE_TRABAJO is not None:
        return _PAQUETE_TRABAJO
    curso = Path(os.environ.get("SystemDrive", "C:") + os.sep) / "curso"
    directos = [Path.cwd().parent / "04_pipeline_sbs",                                   # cuaderno abierto desde 05_Laboratorios del clon
                curso / "trabajo" / "entorno-extraccion-documental" / "04_pipeline_sbs",   # clon en la VM
                curso / "paquete" / "pipeline_sbs"]                                        # respaldo: zip descomprimido
    for candidato in directos:
        if (candidato / "pipeline_sbs" / "__init__.py").is_file():
            _PAQUETE_TRABAJO = candidato
            return candidato
    encontrado = localizar_carpeta("pipeline_sbs", obligatorio=False)      # paquete de material armado (solo lectura)
    if encontrado is not None and (encontrado / "pipeline_sbs" / "__init__.py").is_file():
        _PAQUETE_TRABAJO = _copia_de_trabajo(encontrado)
        return _PAQUETE_TRABAJO
    import importlib.util
    especificacion = importlib.util.find_spec("pipeline_sbs")                 # último recurso: el paquete instalado
    if especificacion is not None and especificacion.origin:
        instalado = Path(especificacion.origin).resolve().parent.parent
        if (instalado / "pipeline_sbs" / "__init__.py").is_file():
            _PAQUETE_TRABAJO = _copia_de_trabajo(instalado)
            return _PAQUETE_TRABAJO
    return None


def importar_pipeline():
    """Importa pipeline_sbs desde el clon (o el respaldo); si no hay carpeta, usa el paquete instalado."""
    raiz = localizar_paquete()
    if raiz is not None:
        sys.path.insert(0, str(raiz))
        for nombre in [n for n in sys.modules if n == "pipeline_sbs" or n.startswith("pipeline_sbs.")]:
            del sys.modules[nombre]
    try:
        import pipeline_sbs
        if not hasattr(pipeline_sbs, "__version__"):
            raise ModuleNotFoundError("pipeline_sbs")
    except ModuleNotFoundError as error:
        raise ModuleNotFoundError(
            "No se encuentra pipeline_sbs. Debe clonarse el repositorio (bloque «Repositorio del docente»): "
            "git clone https://github.com/jonatanfigueroagil-creator/entorno-extraccion-documental.git dentro de "
            "C:\\curso\\trabajo; después se ejecuta de nuevo esta celda. Sin Internet, el respaldo es descomprimir "
            "06_PIPELINE_INICIO_S02.zip en C:\\curso\\paquete."
        ) from error
    return pipeline_sbs


pipeline_sbs = importar_pipeline()


def entorno_pruebas():
    """Variables de entorno para ejecutar pytest sobre el MISMO paquete que usa este cuaderno."""
    raiz = localizar_paquete()
    ruta = os.pathsep.join(p for p in (str(raiz) if raiz else "", os.environ.get("PYTHONPATH", "")) if p)
    return {**os.environ, "SBS_DATOS_PRACTICA": str(DATOS), "PYTHONPATH": ruta}

print("pipeline_sbs", pipeline_sbs.__version__, "· pypdf", pypdf.__version__, "· PyMuPDF", pymupdf.__version__)

pipeline_sbs 0.1.0 · pypdf 6.19.0 · PyMuPDF 1.28.2


### Verificación de la VM: control C00

Esta celda ejecuta el verificador del entorno que viene en la raíz del repositorio clonado (`03_verificar_entorno.py`) con el mismo Python de este cuaderno y muestra su resumen. En palabras simples: es la revisión de la máquina antes de empezar el trabajo, como comprobar que están los equipos y los permisos antes de abrir una oficina. **Debe verse:** una línea `Resumen: N OK · N avisos · 0 errores` y `✔ C00 superado`; los avisos son aceptables. **Si falla:** si el resumen indica errores, si el archivo no existe o si la celda no puede ejecutarlo, avisa y no bloquea el LAB: se avisa al docente o a soporte, y el instalador `01_instalar_vm.ps1` (en la raíz del repositorio clonado) se puede repetir; después se ejecuta de nuevo esta celda.

In [9]:
# ✔ Verificación de la VM (C00)
import subprocess

_paquete = localizar_paquete()
_verificador = (_paquete.parent / "03_verificar_entorno.py") if _paquete is not None else None
if _verificador is None or not _verificador.is_file():
    print("⚠ C00 omitido: no se encuentra 03_verificar_entorno.py en la raíz del repositorio clonado. "
          "Se avisa al docente o a soporte; el instalador 01_instalar_vm.ps1 se puede repetir.")
else:
    try:
        verificacion = subprocess.run([sys.executable, str(_verificador)], capture_output=True, text=True,
                                      encoding="utf-8", errors="replace", timeout=300, cwd=str(_verificador.parent))
    except (OSError, subprocess.TimeoutExpired) as error:
        verificacion = None
        print(f"⚠ C00 no se pudo ejecutar ({type(error).__name__}). Se avisa al docente o a soporte; el instalador 01_instalar_vm.ps1 se puede repetir.")
    if verificacion is not None:
        resumen_vm = [linea for linea in verificacion.stdout.splitlines() if linea.startswith(("Resumen", "[ERROR", "ERROR"))]
        print("\n".join(resumen_vm) if resumen_vm else verificacion.stdout[-600:])
        if verificacion.returncode == 0:
            print("✔ C00 superado: el verificador terminó en 0 errores")
        else:
            print("⚠ C00 con errores: se avisa al docente o a soporte; el instalador 01_instalar_vm.ps1 se puede repetir y luego se ejecuta de nuevo esta celda.")


Resumen: 34 OK · 6 avisos · 0 errores
✔ C00 superado: el verificador terminó en 0 errores


## Su repositorio (8 min): guardar el repositorio del docente en su GitHub

**Contexto:** el participante trabaja sobre una copia del repositorio del docente que está en la VM, y la VM se apaga después de cada sesión sin garantizar que conserve los archivos. Para que su trabajo quede a salvo, esa copia se sube a **su propio repositorio de GitHub** y, al terminar cada ejercicio, se sube de nuevo. En este cuaderno el guardado se hace con una celda `guardar_avance(...)`.

**Antes de empezar (en github.com, en el navegador):** crear un repositorio **privado y vacío** llamado `extraccion-documental-sbs` (sin README, sin `.gitignore` y sin licencia).

**Tarea:** ejecutar en orden tres celdas: (1) las funciones de Git, que no se modifican; (2) completar tres datos (su usuario de GitHub, su nombre y su correo) y ejecutar: identifica al participante, crea su rama `trabajo-s02` y conecta su repositorio con el nombre `mio`; (3) guardar el primer commit y subirlo. La primera vez que sube, se abre una ventana del navegador de la VM para iniciar sesión en GitHub: se inicia sesión con la cuenta con la que se creó el repositorio y se autoriza el acceso.

**En palabras simples:** `commit` guarda una versión en el historial de la VM y `push` la copia a GitHub. Se guarda **por nombre** el cuaderno y el paquete, no los archivos en blanco que JupyterLab crea. Antes de ejecutar una celda de guardado se pulsa `Ctrl+S`, para que el commit lleve lo último que se escribió.

**Debe verse:** `✔ Identidad`, `✔ Rama trabajo-s02`, `✔ Remoto mio` y, en el primer guardado, el último commit con su mensaje y `✔ Subido a mio/trabajo-s02`. En github.com, el repositorio ya no está vacío y muestra la rama `trabajo-s02`.

**Si falla:**
- `Complete USUARIO_GITHUB`: el usuario debe escribirse sin los signos `<` y `>`, sin espacios, sin tildes ni eñes.
- `No se pudo subir` con `Repository not found`: el usuario está mal escrito o el repositorio no se creó; se corrige el dato y se vuelve a ejecutar la celda de configuración.
- `No se pudo subir` con `Authentication failed` o sin que se abra la ventana: se vuelve a ejecutar la celda; si persiste, se avisa al docente.
- `nothing to commit`: no hubo cambios nuevos; se revisa que se pulsó `Ctrl+S`.

In [10]:
# Funciones de Git del LAB (no se modifican)
import subprocess


def _git(*argumentos, tiempo=300):
    """Ejecuta git dentro del repositorio clonado y devuelve (código, texto). No lanza excepciones."""
    paquete = localizar_paquete()
    raiz = paquete.parent if paquete is not None else None
    git = shutil.which("git") or str(Path(os.environ.get("ProgramFiles", "")) / "Git" / "cmd" / "git.exe")
    if raiz is None:
        return 1, "no se encontró el repositorio clonado"
    try:
        r = subprocess.run([git, "-C", str(raiz), *argumentos], capture_output=True, text=True, encoding="utf-8",
                           errors="replace", timeout=tiempo, stdin=subprocess.DEVNULL)
    except (OSError, subprocess.TimeoutExpired) as error:
        return 1, f"{type(error).__name__}: {error}"
    return r.returncode, (r.stdout + r.stderr).strip()


def _es_repositorio():
    codigo_, _ = _git("rev-parse", "--is-inside-work-tree")
    return codigo_ == 0


def configurar_github(usuario, nombre, correo):
    """Identifica al participante, crea la rama trabajo-s02 y conecta su repositorio con el nombre `mio`."""
    usuario, nombre, correo = (usuario or "").strip(), (nombre or "").strip(), (correo or "").strip()
    if not usuario or any(c in usuario for c in "<> ") or not nombre or not correo:
        print("⚠ Complete USUARIO_GITHUB (sin los signos < > ni espacios), NOMBRE y CORREO, y vuelva a ejecutar esta celda.")
        return False
    if not _es_repositorio():
        print("⚠ Este cuaderno no se abrió desde el repositorio clonado: se avisa al docente. Se puede seguir con el LAB sin guardar en GitHub.")
        return False
    _git("config", "user.name", nombre)
    _git("config", "user.email", correo)
    print("✔ Identidad:", _git("config", "user.name")[1], "·", _git("config", "user.email")[1])
    existe, _ = _git("rev-parse", "--verify", "--quiet", "trabajo-s02")
    cod, texto = _git("switch", "trabajo-s02") if existe == 0 else _git("switch", "-c", "trabajo-s02")
    print("✔ Rama trabajo-s02" if cod == 0 else f"⚠ Rama trabajo-s02: {texto}")
    url = f"https://github.com/{usuario}/extraccion-documental-sbs.git"
    hay, _ = _git("remote", "get-url", "mio")
    cod, texto = _git("remote", "set-url", "mio", url) if hay == 0 else _git("remote", "add", "mio", url)
    print(f"✔ Remoto mio = {url}" if cod == 0 else f"⚠ Remoto mio: {texto}")
    return True


def guardar_avance(mensaje, subir=True, etiqueta=None):
    """Guarda el avance: prepara los cambios, crea el commit y, si `subir`, lo sube a `mio`. Con `etiqueta`, la pone y sube las etiquetas."""
    if not _es_repositorio():
        print("⚠ No se puede guardar: el cuaderno no se abrió desde el repositorio clonado. Se avisa al docente; el LAB sigue sin guardar en GitHub.")
        return False
    _git("add", "-A", "--", ".", ":(exclude,glob)**/Untitled*.ipynb")
    cod, texto = _git("commit", "-m", mensaje)
    if cod == 0:
        print("✔ Commit creado:", mensaje)
    elif "nothing to commit" in texto or "nada para" in texto.lower():
        print("ℹ No había cambios nuevos que guardar (¿se pulsó Ctrl+S?). Se sigue con la subida.")
    else:
        print("⚠ No se pudo crear el commit:", texto)
        return False
    if etiqueta:
        _git("tag", "-f", etiqueta)
        print("✔ Etiqueta", etiqueta)
    if subir:
        argumentos = ["push", "-u", "mio", "trabajo-s02"] + (["--tags"] if etiqueta else [])
        cod, texto = _git(*argumentos)
        if cod == 0:
            print("✔ Subido a mio/trabajo-s02")
        else:
            print("⚠ No se pudo subir a GitHub (el commit quedó guardado en la VM):", texto[-400:])
    print(_git("log", "--oneline", "-3")[1])
    return True

In [11]:
# Configurar su repositorio: el participante completa tres datos y ejecuta
USUARIO_GITHUB = "keissipamelalberto-max"   # solución de referencia
NOMBRE = "Keissi Alberto"
CORREO = "keissipamelalberto@gmail.com"

configurar_github(USUARIO_GITHUB, NOMBRE, CORREO)

✔ Identidad: Keissi Alberto · keissipamelalberto@gmail.com
✔ Rama trabajo-s02
✔ Remoto mio = https://github.com/keissipamelalberto-max/extraccion-documental-sbs.git


True

In [ ]:
# Primer commit y subida: guarda el cuaderno del repositorio y lo sube a `mio`
# (antes se pulsa Ctrl+S; la primera vez se abre la ventana de inicio de sesión de GitHub)
guardar_avance("S02: primer commit")

### Verificación: controles C0, C0b y C0c

Esta celda comprueba tres cosas. **C0:** `pytest` pasa sobre el paquete clonado. **C0b:** hay al menos un commit propio, es decir, uno que no está en `origin/main`. **C0c:** el repositorio personal `mio` está configurado y la rama `trabajo-s02` sigue a `mio/trabajo-s02`; la confirmación final se hace en el navegador, en github.com. En palabras simples: C0 confirma que la caja de herramientas llegó completa, C0b que el primer cambio quedó guardado en el historial y C0c que la copia personal en GitHub está conectada. **Debe verse:** una línea `N passed`, el historial de commits y los mensajes `✔ C0 superado`, `✔ C0b superado` y `✔ C0c`. **Si falla:** en C0, el mensaje de `pytest` indica qué prueba revisar y se repite `pip install -e .\04_pipeline_sbs`; en C0b, si falta el commit propio se vuelve a ejecutar la celda «Primer commit y subida»; en C0c, si falta el remoto se vuelve a ejecutar la celda «Configurar su repositorio» y después la del primer commit. Si Git no está en el PATH o la carpeta no es un clon, la celda avisa y continúa: no bloquea el LAB.

In [12]:
# ✔ Verificación (C0, C0b y C0c)
import subprocess

RAIZ_PAQUETE = localizar_paquete()
carpeta_pruebas = (RAIZ_PAQUETE / "pipeline_sbs" / "tests") if RAIZ_PAQUETE else (Path(pipeline_sbs.__file__).parent / "tests")
corrida = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(carpeta_pruebas)],
    capture_output=True, text=True, env=entorno_pruebas(), timeout=300,
)
print(corrida.stdout.strip().splitlines()[-1] if corrida.stdout.strip() else corrida.stderr[-300:])
assert corrida.returncode == 0, "C0: pytest falla. Se revisa el mensaje de arriba y se repite pip install -e .\\04_pipeline_sbs."
print("✔ C0 superado: las pruebas del paquete pasan")


def ejecutar_git(*argumentos):
    """Ejecuta git dentro del clon; devuelve None si Git no está disponible."""
    git = shutil.which("git") or str(Path(os.environ.get("ProgramFiles", "")) / "Git" / "cmd" / "git.exe")
    try:
        return subprocess.run([git, "-C", str(RAIZ_PAQUETE), *argumentos], capture_output=True, text=True, timeout=30)
    except (OSError, subprocess.TimeoutExpired):
        return None


def mostrar_ultimo_commit(cantidad: int = 3) -> None:
    """Muestra los últimos commits y si la rama tiene cambios sin subir; avisa sin detener el cuaderno si Git no está."""
    historial = ejecutar_git("log", "--oneline", f"-{cantidad}") if RAIZ_PAQUETE else None
    if historial is None or historial.returncode != 0:
        print("⚠ No se pudo leer `git log` (Git no está en el PATH o la carpeta no es un clon). No bloquea el LAB.")
        return
    print(historial.stdout.strip())
    estado = ejecutar_git("status", "-sb")
    if estado is not None and estado.returncode == 0:
        print(estado.stdout.splitlines()[0])


historial = ejecutar_git("log", "--oneline") if RAIZ_PAQUETE else None
if historial is None or historial.returncode != 0:
    print("⚠ C0b omitido: no se pudo leer `git log` (Git no está en el PATH o la carpeta no es un clon). Se avisa al docente.")
else:
    print(historial.stdout)
    propios = ejecutar_git("rev-list", "--count", "origin/main..HEAD")
    if propios is not None and propios.returncode == 0:
        commits_propios = int(propios.stdout.strip())
    else:
        print("⚠ No se encontró origin/main: se cuentan los commits de la rama menos el de partida.")
        commits_propios = len(historial.stdout.splitlines()) - 1
    assert commits_propios >= 1, "C0b: no hay un commit propio. Se ejecuta la celda «Primer commit y subida» de la sección «Su repositorio»."
    print(f"✔ C0b superado: {commits_propios} commit(s) propio(s) sobre origin/main")

remoto = ejecutar_git("remote", "get-url", "mio") if RAIZ_PAQUETE else None
if remoto is None or remoto.returncode != 0:
    print("⚠ C0c pendiente: no existe el remoto `mio`. Se ejecuta la celda «Configurar su repositorio»; no bloquea el LAB.")
else:
    seguimiento = ejecutar_git("rev-parse", "--abbrev-ref", "@{u}")
    if seguimiento is None or seguimiento.returncode != 0:
        print("⚠ C0c pendiente: la rama no sigue a `mio`. Se ejecuta la celda «Primer commit y subida»; no bloquea el LAB.")
    else:
        print(f"✔ C0c: remoto mio = {remoto.stdout.strip()} · la rama sigue a {seguimiento.stdout.strip()}. Falta confirmar la rama en github.com.")

31 passed in 1.67s
✔ C0 superado: las pruebas del paquete pasan
74d705c S02: primer commit
469a7db LAB S02: la pista del hueco del duplicado incluye el detalle del registro
4c39d53 LAB S02: los avisos de C7 indican volver a ejecutar la celda Guardar, no comandos de Git
8e4e8cd LAB S02: el guardado en GitHub se hace desde el cuaderno (configurar_github y guardar_avance) en cada ejercicio; C0b cuenta los commits que no estan en origin/main
fc17cfb gitignore: no se versiona la carpeta .ipynb_checkpoints que crea JupyterLab
90818fb LAB S02: 22 celdas de texto tenian tipo md (invalido) y JupyterLab las mostraba como texto crudo; ahora son markdown
1cf0a01 Instalador: la falta del controlador NVIDIA se informa como aviso (se necesita desde la sesion 4)
4b3b365 Instalador: pip muestra su avance (Collecting y Successfully) en vez de callar hasta el final
4c9bb71 Ollama y su modelo pasan a ser opcionales (-ConOllama, desde la sesion 5); el verificador los informa como aviso salvo con --con-llm


## Reto de apertura (6 min) — Ingesta · M2.1: ¿Cuántos de estos 20 archivos abre un programa sin protecciones?

**Contexto:** la Mesa de Partes de *Financiera Altiplano DEMO S.A.* (entidad ficticia) dejó 20 archivos mixtos (PDF, DOCX, correos, HTML e imágenes) y necesita cargarlos, como cualquier mesa de partes que recibe de una vez oficios, estados financieros y contratos.

**Insumo:** `02_Datos_Practica/01_Lote_S02/` (LOT-0001 a LOT-0020).

**Tarea:** el participante completa `abrir(ruta)`, que abre cada archivo según su extensión **sin** `try/except`, observa dónde se detiene un programa simple y anota en qué archivo se cae, con qué mensaje de error (la excepción) y cuántos alcanzó a abrir.

**Debe verse:** una lista que avanza y, de pronto, un mensaje de error con el nombre del archivo en el que se cayó. **Si falla o tarda:** que el programa se caiga es justamente lo que se quiere ver; si una celda pasa de 10 minutos, se interrumpe y se usa el subconjunto de 10 archivos o se avisa al docente.

In [13]:
LOTE = DATOS / "01_Lote_S02"
archivos_lote = sorted(p for p in LOTE.iterdir() if p.is_file())
print(len(archivos_lote), "archivos en el lote")


def abrir(ruta):
    """Abre el archivo con la biblioteca de su extensión. No captura errores."""
    ext = ruta.suffix.lower()
    if ext == ".pdf":
        return pypdf.PdfReader(str(ruta))
    if ext == ".docx":
        return Document(str(ruta))
    if ext in (".eml", ".html"):
        return ruta.read_text(encoding="utf-8")
    if ext in (".png", ".jpg"):
        with Image.open(ruta) as imagen:
            imagen.load()
        return imagen
    raise ValueError(f"Extensión no prevista: {ext}")

20 archivos en el lote


In [14]:
abiertos = 0
caida = None
for ruta in archivos_lote:
    try:
        abrir(ruta)
        abiertos += 1
    except Exception as error:  # solo para que el notebook siga: un script sin protección se detiene aquí
        caida = (ruta.name, type(error).__name__)
        break

print(f"Abrió {abiertos} de {len(archivos_lote)} archivos antes de caerse.")
print("Se cayó en:", caida)

mi_anotacion = {"archivo": caida[0], "excepcion": caida[1], "abiertos": abiertos}

invalid pdf header: b'\x89PNG\r'
EOF marker not found
EOF marker not found
EOF marker not found


Abrió 1 de 20 archivos antes de caerse.
Se cayó en: ('LOT-0002.pdf', 'PdfStreamError')


In [15]:
# ✔ Checkpoint del reto
assert abiertos < len(archivos_lote) and caida is not None, "El bucle debería detenerse en algún archivo."
assert mi_anotacion == {"archivo": caida[0], "excepcion": caida[1], "abiertos": abiertos}, \
    f"La anotación debe coincidir con la salida: {caida} y {abiertos} abiertos."
print(f"✔ Reto superado: el script abrió {abiertos} de 20 y se cayó en {caida[0]} con {caida[1]}")

✔ Reto superado: el script abrió 1 de 20 y se cayó en LOT-0002.pdf con PdfStreamError


## Ejercicio 1 (15 min) — Lectura · M2.2, M2.4: Un lector por formato

**Contexto:** *Financiera Altiplano DEMO S.A.* recibe un estado financiero, un oficio escaneado, un contrato, un reclamo por correo, un tarifario web y un voucher fotografiado. El participante escribe un lector corto para cada formato y decide cuál archivo necesita OCR.

**Insumo:** `03_Estados_Financieros/EFI-0001.pdf`, `02_Oficios/OFI-0003.pdf`, `04_Contratos/CON-0001.docx`, `06_Reclamos/REC-0001.eml`, `07_Tarifarios/TAR-0001.html` y `08_Vouchers/VOU-0001.jpg` (dentro de `02_Datos_Practica`).

**Tarea:** el participante completa los huecos marcados `# TODO` de cinco lectores (3 a 6 líneas cada uno): PDF con las tres bibliotecas (pypdf, pdfplumber y PyMuPDF), DOCX (python-docx), correo (`email`), HTML (BeautifulSoup) e imagen (Pillow, solo apertura y dimensiones). La medición de las tres bibliotecas de PDF y la tabla de formatos vienen escritas: se ejecutan y se leen. El OCR de la imagen queda de referencia y no se ejecuta.

**En palabras simples:** un estado financiero en PDF suele traer el texto adentro; un oficio firmado y escaneado es una foto de la hoja y no tiene letras que copiar: para leerlo hace falta OCR, un programa que reconoce letras en una imagen. Una biblioteca es un programa auxiliar que ya sabe leer un formato. **Debe verse:** en `EFI-0001`, más de 1 000 caracteres con las tres bibliotecas; en `OFI-0003` y en la imagen, 0 caracteres y sin error. **Si falla:** con `fitz` se escribe `import pymupdf`; un `AssertionError` del checkpoint indica qué lector revisar y no es una falla del participante.

> ◆ **Píldora P1: Qué librería usar y cuándo (3 min).** *Pregunta de supervisión: ¿qué programa lee bien un estado financiero o un oficio en PDF y se puede usar sin problemas legales?* Las tres bibliotecas extraen igual de un PDF con texto y devuelven vacío de un escaneado, sin error; lo que cambia es la licencia: PyMuPDF es AGPL o comercial, pypdf BSD y pdfplumber MIT.
> La AGPL obliga a abrir el código de lo que se construya con ella, por eso se consulta antes de usarla en un servicio de la institución. El curso fija PyMuPDF 1.28.2 con `import pymupdf`.

In [16]:
# Contrato de datos y utilidades de E1 (no se modifica)
@dataclass(frozen=True)
class Extraccion:
    """Resultado de extraer un archivo: texto, páginas y caracteres de cada página."""

    texto: str
    paginas: int
    caracteres_por_pagina: tuple = ()
    metadatos: dict = field(default_factory=dict)

    @property
    def caracteres(self) -> int:
        return len(self.texto.strip())


def extraccion_desde_paginas(textos_por_pagina):
    """Arma una Extraccion a partir del texto de cada página."""
    return Extraccion(
        texto="\n".join(textos_por_pagina),
        paginas=len(textos_por_pagina),
        caracteres_por_pagina=tuple(len(t.strip()) for t in textos_por_pagina),
    )


def medir(funcion, ruta):
    """Ejecuta funcion(ruta) y devuelve (resultado, segundos) medidos con time.perf_counter()."""
    inicio = time.perf_counter()
    resultado = funcion(ruta)
    return resultado, time.perf_counter() - inicio


def decodificar_html(contenido: bytes) -> str:
    """Convierte bytes de HTML en texto: BOM, charset declarado, UTF-8 y, al final, cp1252."""
    if contenido.startswith(b"\xef\xbb\xbf"):
        return contenido.decode("utf-8-sig")
    declarado = re.search(rb"charset\s*=\s*[\"']?\s*([A-Za-z0-9_\-]+)", contenido[:2048], re.IGNORECASE)
    if declarado:
        try:
            return contenido.decode(declarado.group(1).decode("ascii"))
        except (LookupError, UnicodeDecodeError):
            pass
    try:
        return contenido.decode("utf-8")
    except UnicodeDecodeError:
        return contenido.decode("cp1252", errors="replace")


def filas_de_tabla(tabla):
    """Filas de una tabla de Word como texto 'celda | celda', sin repetir las celdas combinadas."""
    filas = []
    for fila in tabla.rows:
        vistas, celdas = set(), []
        for celda in fila.cells:
            if id(celda._tc) in vistas:
                continue
            vistas.add(id(celda._tc))
            celdas.append(celda.text.strip())
        filas.append(" | ".join(celdas))
    return filas


RUTA = {
    "EFI-0001": DATOS / "03_Estados_Financieros" / "EFI-0001.pdf",
    "OFI-0003": DATOS / "02_Oficios" / "OFI-0003.pdf",
    "CON-0001": DATOS / "04_Contratos" / "CON-0001.docx",
    "REC-0001": DATOS / "06_Reclamos" / "REC-0001.eml",
    "TAR-0001": DATOS / "07_Tarifarios" / "TAR-0001.html",
    "VOU-0001": DATOS / "08_Vouchers" / "VOU-0001.jpg",
}

In [17]:
# Lectores de PDF con tres bibliotecas
def extraer_pdf_pypdf(ruta):
    """Texto de un PDF con pypdf (BSD-3), una entrada por página."""
    lector = pypdf.PdfReader(str(ruta))
    return extraccion_desde_paginas([pagina.extract_text() or "" for pagina in lector.pages])


def extraer_pdf_pdfplumber(ruta):
    """Texto de un PDF con pdfplumber (MIT), una entrada por página."""
    with pdfplumber.open(str(ruta)) as pdf:
        textos = [pagina.extract_text() or "" for pagina in pdf.pages]
    return extraccion_desde_paginas(textos)


def extraer_pdf_pymupdf(ruta):
    """Texto de un PDF con PyMuPDF (AGPL o comercial), una entrada por página."""
    with pymupdf.open(str(ruta)) as doc:
        textos = [pagina.get_text() for pagina in doc]
    return extraccion_desde_paginas(textos)

In [18]:
# Lectores de DOCX, correo, HTML e imagen
def extraer_docx(ruta):
    """Párrafos y tablas de nivel superior de un .docx, en el orden del documento."""
    documento = Document(str(ruta))
    bloques = []
    for hijo in documento.element.body.iterchildren():      # recorre el cuerpo en el orden del documento
        if hijo.tag.endswith("}p"):
            bloques.append(Paragraph(hijo, documento).text)
        elif hijo.tag.endswith("}tbl"):
            bloques.extend(filas_de_tabla(Table(hijo, documento)))
    texto = "\n".join(bloques)
    return Extraccion(texto=texto, paginas=1, caracteres_por_pagina=(len(texto.strip()),),
                      metadatos={"tablas": len(documento.tables)})


def extraer_eml(ruta):
    """Cuerpo de un correo .eml y nombres de sus adjuntos."""
    with open(ruta, "rb") as archivo:
        mensaje = BytesParser(policy=policy.default).parse(archivo)
    cuerpo = mensaje.get_body(preferencelist=("plain", "html"))
    texto = cuerpo.get_content() if cuerpo is not None else ""
    adjuntos = [Path(p.get_filename() or "sin_nombre").name for p in mensaje.iter_attachments()]
    return Extraccion(texto=texto, paginas=1, caracteres_por_pagina=(len(texto.strip()),),
                      metadatos={"asunto": str(mensaje.get("Subject", "")), "adjuntos": adjuntos})


def extraer_html(ruta):
    """Texto visible de un HTML, sin script ni estilo."""
    sopa = BeautifulSoup(decodificar_html(Path(ruta).read_bytes()), "html.parser")
    for etiqueta in sopa(["script", "style", "template"]):
        etiqueta.decompose()
    texto = sopa.get_text("\n", strip=True)
    return Extraccion(texto=texto, paginas=1, caracteres_por_pagina=(len(texto.strip()),))


def extraer_imagen(ruta):
    """Una imagen no tiene texto seleccionable: texto vacío, una página y su tamaño."""
    with Image.open(ruta) as imagen:
        ancho, alto = imagen.size
    return Extraccion(texto="", paginas=1, caracteres_por_pagina=(0,), metadatos={"ancho": ancho, "alto": alto})

**OCR de referencia (no se ejecuta en esta sesión).** Una imagen se abre y se mide, pero no entrega texto: para leerla hace falta un OCR. PaddleOCR se ejecuta en la sesión 3, en la VM; en este ejercicio el lector de imagen solo comprueba que la imagen abre y registra su ancho y alto.

In [19]:
BIBLIOTECAS_PDF = {"pypdf": extraer_pdf_pypdf, "pdfplumber": extraer_pdf_pdfplumber, "pymupdf": extraer_pdf_pymupdf}

filas = []
for doc_id in ("EFI-0001", "OFI-0003"):
    for biblioteca, funcion in BIBLIOTECAS_PDF.items():
        resultado, segundos = medir(funcion, RUTA[doc_id])
        filas.append({"documento": doc_id, "biblioteca": biblioteca, "paginas": resultado.paginas,
                      "caracteres": resultado.caracteres, "segundos": segundos})

tabla_pdf = pd.DataFrame(filas)
print(tabla_pdf.to_string(index=False))

documento biblioteca  paginas  caracteres  segundos
 EFI-0001      pypdf        2        1232  0.020364
 EFI-0001 pdfplumber        2        1231  0.060181
 EFI-0001    pymupdf        2        1232  0.028490
 OFI-0003      pypdf        1           0  0.002526
 OFI-0003 pdfplumber        1           0  0.002617
 OFI-0003    pymupdf        1           0  0.002713


In [20]:
UMBRAL_OCR = 20   # caracteres por página por debajo de los cuales un PDF requiere OCR

por_pagina = tabla_pdf.assign(pp=tabla_pdf["caracteres"] / tabla_pdf["paginas"]).groupby("documento")["pp"].max()
requieren_ocr = set(por_pagina[por_pagina < UMBRAL_OCR].index)
nativo = tabla_pdf[tabla_pdf["documento"] == "EFI-0001"]
mas_rapida = nativo.loc[nativo["segundos"].idxmin(), "biblioteca"]
print("Requieren OCR:", requieren_ocr, "· la más rápida en el PDF nativo:", mas_rapida)

Requieren OCR: {'OFI-0003'} · la más rápida en el PDF nativo: pypdf


In [21]:
OTROS = {"CON-0001": extraer_docx, "REC-0001": extraer_eml, "TAR-0001": extraer_html, "VOU-0001": extraer_imagen}

filas_otros = []
for doc_id, funcion in OTROS.items():
    resultado, segundos = medir(funcion, RUTA[doc_id])
    filas_otros.append({"documento": doc_id, "formato": RUTA[doc_id].suffix.lstrip("."),
                        "caracteres": resultado.caracteres, "segundos": segundos})

tabla_otros = pd.DataFrame(filas_otros)
print(tabla_otros.to_string(index=False))

documento formato  caracteres  segundos
 CON-0001    docx       19925  0.057949
 REC-0001     eml         264  0.026706
 TAR-0001    html         690  0.022759
 VOU-0001     jpg           0  0.011183


In [22]:
# ✔ Checkpoint 1
assert len(tabla_pdf) == 6, "Se miden 2 documentos x 3 bibliotecas = 6 filas."
nativo = tabla_pdf[tabla_pdf["documento"] == "EFI-0001"]
assert (nativo["caracteres"] > 1000).all(), "El PDF nativo debe dar más de 1000 caracteres con las tres bibliotecas: se revisan los tres lectores de PDF."
assert nativo["caracteres"].max() - nativo["caracteres"].min() <= 0.05 * nativo["caracteres"].max(), \
    "Las tres bibliotecas deben coincidir (±5 %) en un PDF con texto."
assert (tabla_pdf[tabla_pdf["documento"] == "OFI-0003"]["caracteres"] == 0).all(), \
    "El escaneado da 0 caracteres con las tres bibliotecas, y sin error."
assert requieren_ocr == {"OFI-0003"}, f"requieren_ocr = {requieren_ocr}"
assert (tabla_pdf["segundos"] > 0).all(), "Los segundos se miden con time.perf_counter()."
assert mas_rapida in BIBLIOTECAS_PDF and mas_rapida == nativo.loc[nativo["segundos"].idxmin(), "biblioteca"]
otros = tabla_otros.set_index("documento")
assert len(otros) == 4, "Faltan formatos en tabla_otros."
assert otros.loc["CON-0001", "caracteres"] == 19925, "El lector de DOCX debe incluir párrafos Y tablas (19925 caracteres): se revisa extraer_docx."
assert otros.loc["REC-0001", "caracteres"] > 0 and extraer_eml(RUTA["REC-0001"]).metadatos["asunto"], "El lector de correo debe devolver el cuerpo: se revisa extraer_eml."
assert otros.loc["TAR-0001", "caracteres"] > 0 and "<" not in extraer_html(RUTA["TAR-0001"]).texto, "El lector de HTML debe devolver el texto visible: se revisa extraer_html."
dimensiones = extraer_imagen(RUTA["VOU-0001"]).metadatos
assert dimensiones.get("ancho", 0) > 0 and dimensiones.get("alto", 0) > 0, "El lector de imagen debe devolver ancho y alto: se revisa extraer_imagen."
assert otros.loc["VOU-0001", "caracteres"] == 0, "La imagen no tiene texto seleccionable: requiere OCR (S03)."
print("✔ Checkpoint 1 superado: cinco lectores; el escaneado y la imagen abrieron sin error, pero no entregan texto: requieren OCR")

✔ Checkpoint 1 superado: cinco lectores; el escaneado y la imagen abrieron sin error, pero no entregan texto: requieren OCR


### Guardar (≈1 min)

El participante pulsa `Ctrl+S` y ejecuta la celda de abajo: crea el commit en la VM y lo sube a `mio`, su repositorio personal (nunca a `origin`). **Debe verse:** `✔ Commit creado`, `✔ Subido a mio/trabajo-s02` y el último commit arriba de la lista. En github.com, el repositorio muestra el commit nuevo con ese mensaje.

In [23]:
# Guardar el ejercicio: commit y subida a `mio`
guardar_avance("S02: E1 un lector por formato")

✔ Commit creado: S02: E1 un lector por formato
✔ Subido a mio/trabajo-s02
71abaad S02: E1 un lector por formato
74d705c S02: primer commit
469a7db LAB S02: la pista del hueco del duplicado incluye el detalle del registro


True

## Ejercicio 2 (8 min) — Lectura · M2.4: Tablas de un PDF con texto, tabula-py frente a Camelot

**Contexto:** el área de riesgos de *Financiera Altiplano DEMO S.A.* quiere leer el balance de un estado financiero como tabla para validar que el total de activo cuadra.

**Insumo:** `03_Estados_Financieros/EFI-0001.pdf` (página 1, tabla sin bordes) y su respuesta esperada en `10_Ground_Truth/EFI-0001.json`.

**Tarea:** el participante lee la tabla con tabula-py y con Camelot (modo `stream`, porque la tabla no tiene bordes) y compara filas, columnas y total de activo con la respuesta esperada. Las funciones de lectura y la comparación vienen escritas; el participante completa `leer_total_activo` y la decisión de qué herramienta usaría. Si falta Java o alguna biblioteca, el plan B (lectura por líneas con pdfplumber) ya está escrito.

**En palabras simples:** el total de activo es la cifra que el supervisor revisa primero para ver si el balance cuadra. Camelot lee tablas de dos maneras: `lattice` busca las líneas dibujadas y `stream` se guía por la alineación del texto; este balance no tiene líneas, por eso se usa `stream`. Java es un programa de apoyo que tabula-py necesita. **Debe verse:** cada herramienta devuelve una tabla y el total de activo coincide con la respuesta esperada. **Si falla:** si `java -version` no responde, no es un error del participante: se avisa al docente y se trabaja con el plan B.

In [24]:
# Disponibilidad de herramientas y plan B (no se modifica)
HAY_JAVA = shutil.which("java") is not None
try:
    import tabula
    HAY_TABULA = HAY_JAVA
except ImportError:
    HAY_TABULA = False
try:
    import camelot
    HAY_CAMELOT = True
except ImportError:
    HAY_CAMELOT = False
print("Java:", HAY_JAVA, "· tabula-py:", HAY_TABULA, "· Camelot:", HAY_CAMELOT)
if not (HAY_TABULA and HAY_CAMELOT):
    print("Falta alguna herramienta: se comparan las disponibles junto con el plan B.")


def tabla_por_lineas(ruta, pagina=0):
    """Plan B sin Java ni Camelot: separa por la derecha cada línea 'Concepto 1,234 5,678'."""
    with pdfplumber.open(str(ruta)) as pdf:
        lineas = (pdf.pages[pagina].extract_text() or "").splitlines()
    filas, dentro = [], False
    for linea in lineas:
        dentro = dentro or linea.startswith("Concepto")
        if not dentro:
            continue
        partes = linea.rsplit(" ", 2)
        if len(partes) == 3 and all(re.fullmatch(r"[\d,]+", p) for p in partes[1:]):
            filas.append(partes)
        else:
            filas.append([linea, "", ""])
    return pd.DataFrame(filas)

Java: True · tabula-py: True · Camelot: True


In [25]:
def leer_total_activo(tabla):
    """Total de activo (entero) de una tabla, o None si la tabla no trae esa fila."""
    for _, fila in tabla.iterrows():
        if str(fila.iloc[0]).strip() == "Total activo":
            return int(str(fila.iloc[1]).replace(",", ""))
    return None


def tablas_tabula(ruta):
    """Lista de DataFrame que devuelve tabula-py en la página 1."""
    return tabula.read_pdf(str(ruta), pages=1, stream=True, silent=True, pandas_options={"header": None})


def tablas_camelot(ruta):
    """Lista de DataFrame que devuelve Camelot (modo stream) en la página 1."""
    return [t.df for t in camelot.read_pdf(str(ruta), pages="1", flavor="stream")]

In [26]:
VERDAD = json.loads((DATOS / "10_Ground_Truth" / "EFI-0001.json").read_text(encoding="utf-8"))
filas_esperadas = len(VERDAD["tablas"][0]["filas"]) + 1      # +1 por el encabezado
total_esperado = VERDAD["campos"]["total_activo"]

candidatas = {"plan B (pdfplumber)": lambda ruta: [tabla_por_lineas(ruta)]}
if HAY_TABULA:
    candidatas["tabula-py"] = tablas_tabula
if HAY_CAMELOT:
    candidatas["Camelot"] = tablas_camelot

comparacion = []
for nombre, funcion in candidatas.items():
    tablas, segundos = medir(funcion, RUTA["EFI-0001"])
    con_total = [t for t in tablas if leer_total_activo(t) is not None]
    elegida = con_total[0] if con_total else None
    comparacion.append({
        "herramienta": nombre, "tablas": len(tablas),
        "filas": len(elegida) if elegida is not None else 0,
        "columnas": elegida.shape[1] if elegida is not None else 0,
        "total_leido": leer_total_activo(elegida) if elegida is not None else None,
        "segundos": round(segundos, 3),
    })

tabla_comparacion = pd.DataFrame(comparacion)
if not tabla_comparacion.empty:
    tabla_comparacion["filas_ok"] = tabla_comparacion["filas"] == filas_esperadas
    tabla_comparacion["total_ok"] = tabla_comparacion["total_leido"] == total_esperado
print(f"Respuesta esperada: {filas_esperadas} filas · total de activo {total_esperado}")
print(tabla_comparacion.to_string(index=False))

decision = ("Conviene tabula-py en modo stream cuando hay Java: devuelve la tabla completa en una sola pieza (22 filas) con el "
            "total correcto; Camelot parte la página en varias tablas y exige revisar. Sin Java, el plan B por líneas da el mismo total.")

Failed to import jpype dependencies. Fallback to subprocess.
No module named 'jpype'


Respuesta esperada: 22 filas · total de activo 39317
        herramienta  tablas  filas  columnas  total_leido  segundos  filas_ok  total_ok
plan B (pdfplumber)       1     22         3        39317     0.147      True      True
          tabula-py       1     22         3        39317     2.823      True      True
            Camelot       3     12         3        39317     0.033     False      True


C:\curso\venv\Lib\site-packages\camelot\parsers\base.py:302: UserWarning: No tables found in table area (40.0, 724.0269999999999, 408.6599999999999, 847.745875)
  cols, rows, v_s, h_s = self._generate_columns_and_rows(bbox, user_cols)


In [27]:
# ✔ Checkpoint 2
assert not tabla_comparacion.empty, "Falta completar el TODO de la comparación: debe haber al menos una herramienta."
assert (tabla_comparacion["tablas"] >= 1).all(), "Cada herramienta debe devolver al menos una tabla (DataFrame)."
assert tabla_comparacion["total_ok"].all(), f"El total de activo leído debe ser {total_esperado}: {tabla_comparacion['total_leido'].tolist()}"
assert len(decision.split()) >= 15, "La decisión se justifica con al menos 15 palabras."
print("✔ Checkpoint 2 superado:", ", ".join(tabla_comparacion["herramienta"]))

✔ Checkpoint 2 superado: plan B (pdfplumber), tabula-py, Camelot


### Guardar (≈1 min)

El participante pulsa `Ctrl+S` y ejecuta la celda de abajo: crea el commit en la VM y lo sube a `mio`, su repositorio personal (nunca a `origin`). **Debe verse:** `✔ Commit creado`, `✔ Subido a mio/trabajo-s02` y el último commit arriba de la lista. En github.com, el repositorio muestra el commit nuevo con ese mensaje.

In [ ]:
# Guardar el ejercicio: commit y subida a `mio`
guardar_avance("S02: E2 tablas")

## Ejercicio 3 (10 min) — Limpieza y normalización · M2.3: Funciones de normalización con pruebas

**Contexto:** el texto que sale de los lectores no es uniforme: espacios raros, palabras cortadas, fechas y montos en soles escritos de varias formas. *Financiera Altiplano DEMO S.A.* necesita funciones puras y probadas antes de comparar o buscar texto.

**Insumo:** las cadenas sintéticas de la celda siguiente (inyectadas a propósito) y el texto de `03_Estados_Financieros/EFI-0001.pdf`.

**Tarea:** el participante completa los huecos de `normalizar_texto` (NFC, espacios especiales y espacios repetidos) y de `parsear_monto` (un solo tipo de separador); `unir_guiones` y `parsear_fecha` vienen escritas. El checkpoint las pone a prueba, incluida la prueba que muestra por qué no se usa NFKC.

**En palabras simples:** normalizar es dejar el texto en una sola forma para que `S/ 1 250,00` y `S/ 1,250.00` de dos informes distintos sean el mismo monto, y para que el «Nº 012» de un oficio no pierda su símbolo. NFC y NFKC son dos maneras de unificar letras con tilde. **Debe verse:** `✔ Checkpoint 3 superado` con el total de activo de `EFI-0001`. **Si falla:** un `AssertionError` es la comprobación señalando qué revisar, no una falla del participante: si menciona NFC, se usó NFKC; si menciona el guion, se revisa la regla del guion de corte.

> ◆ **Píldora P2: Limpieza y normalización (3 min).** *Pregunta de supervisión: si dos oficios dicen lo mismo pero el sistema no los reconoce como iguales, ¿qué está pasando con el texto?* Dos textos que se ven iguales pueden ser distintos por dentro hasta aplicar NFC, que junta la letra y su tilde; NFKC limpia de más y convierte `Nº` en `No`. Conviene normalizar poco y con pruebas.
> ```python
> unicodedata.normalize("NFC", "café") == "café"   # True: "café" es la e con el acento aparte
> unicodedata.normalize("NFKC", "Nº 123")          # 'No 123': cambió el contenido
> ```

In [ ]:
# Cadenas sintéticas inyectadas y constantes (no se modifican)
ESPACIOS_ESPECIALES = "\u00a0\u1680\u2000\u2001\u2002\u2003\u2004\u2005\u2006\u2007\u2008\u2009\u200a\u202f\u205f\u3000"
INVISIBLES = "\u00ad\u200b\u200c\u200d\ufeff"          # guion blando, ancho cero, marca de orden de bytes
LIGADURAS = {"\ufb00": "ff", "\ufb01": "fi", "\ufb02": "fl", "\ufb03": "ffi", "\ufb04": "ffl"}

CASOS = {
    "nfd": "café financiero",
    "espacios": "S/ 1 250,00",
    "guion": "la informa-\ncion recibida",
}
print("¿'café' == 'café'?", "café" == "café", "· ¿se ven iguales?", "café", "café")

In [ ]:
def normalizar_texto(texto):
    """NFC, caracteres invisibles fuera, espacios especiales cambiados por un espacio y espacios repetidos colapsados."""
    texto = unicodedata.normalize("NFC", texto)
    for invisible in INVISIBLES:
        texto = texto.replace(invisible, "")
    for ligadura, reemplazo in LIGADURAS.items():
        texto = texto.replace(ligadura, reemplazo)
    for especial in ESPACIOS_ESPECIALES:
        texto = texto.replace(especial, " ")
    texto = texto.replace("\r\n", "\n").replace("\r", "\n")
    lineas = [re.sub(r"[ \t]+", " ", linea).strip() for linea in texto.split("\n")]
    return "\n".join(lineas).strip()


def unir_guiones(texto):
    """Une las palabras cortadas con guion al final de línea."""
    return re.sub(r"(?<=[a-záéíóúüñ])-\n(?=[a-záéíóúüñ])", "", texto)


FORMATOS_FECHA = ("%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d")


def parsear_fecha(texto):
    """Fecha (date) o None si no es una fecha válida."""
    for formato in FORMATOS_FECHA:
        try:
            return datetime.strptime(texto.strip(), formato).date()
        except ValueError:
            continue
    return None


def parsear_monto(texto):
    """Monto en soles como Decimal, o None si no hay un número."""
    limpio = normalizar_texto(texto).replace("S/", "").replace(" ", "")
    if not re.search(r"\d", limpio):
        return None
    if "," in limpio and "." in limpio:
        decimal = "," if limpio.rfind(",") > limpio.rfind(".") else "."     # el separador que aparece último es el decimal
        miles = "." if decimal == "," else ","
        limpio = limpio.replace(miles, "").replace(decimal, ".")
    elif "," in limpio or "." in limpio:
        separador = "," if "," in limpio else "."
        partes = limpio.split(separador)
        limpio = "".join(partes) if len(partes) > 2 or len(partes[-1]) == 3 else ".".join(partes)
    return Decimal(limpio)

In [ ]:
# ✔ Checkpoint 3
assert normalizar_texto(CASOS["nfd"]) == "café financiero", "NFC debe unir la e y el acento combinado."
assert normalizar_texto(CASOS["espacios"]) == "S/ 1 250,00", "NBSP y el espacio fino se cambian por un espacio normal."
assert normalizar_texto("  muchos    espacios  ") == "muchos espacios"
assert unicodedata.normalize("NFKC", "Oficio Nº 123") == "Oficio No 123"      # NFKC cambia el contenido
assert normalizar_texto("Oficio Nº 123") == "Oficio Nº 123", "No debe usarse NFKC: cambia Nº por No."
assert unir_guiones(CASOS["guion"]) == "la informacion recibida"
assert unir_guiones("plazo - 15 dias") == "plazo - 15 dias"
assert unir_guiones("socio-\neconomico") == "socioeconomico"                    # limitación: también une compuestos legítimos
assert parsear_fecha("11/08/2026") == date(2026, 8, 11) and parsear_fecha("2026-08-11") == date(2026, 8, 11)
assert parsear_fecha("31/02/2026") is None and parsear_fecha("sin fecha") is None
assert parsear_monto(CASOS["espacios"]) == Decimal("1250.00")
assert parsear_monto("S/ 8,861.00") == Decimal("8861.00") and parsear_monto("1.250,50") == Decimal("1250.50")
assert parsear_monto("4,143") == Decimal("4143")
assert parsear_monto("sin monto") is None

texto_efi = normalizar_texto(extraer_pdf_pymupdf(RUTA["EFI-0001"]).texto)
total_texto = texto_efi.split("Total activo")[1].split()[0]
assert parsear_monto(total_texto) == Decimal(VERDAD["campos"]["total_activo"]), "El total del PDF debe coincidir con la respuesta esperada."
print("✔ Checkpoint 3 superado: el total de activo del PDF es", parsear_monto(total_texto))

### Reto de ampliación (opcional, sin nota)

**Contexto:** el área de riesgos de *Financiera Altiplano DEMO S.A.* recibe montos y fechas escritos de formas distintas según la oficina de origen. Para los participantes que ya dominan la limpieza, el reto es normalizar esos formatos hacia `Decimal` y `date`.

**Insumo:** los 5 casos de prueba de la celda siguiente y las funciones `parsear_monto` y `parsear_fecha` que el participante acaba de completar.

**Tarea:** el participante comprueba que `parsear_monto` ya cubre `S/ 1 250,00`, `1,250.50` y `1.250,5`. `parsear_fecha` no reconoce meses escritos en texto, por lo que se amplía **dentro del cuaderno** con la función `parsear_fecha_ampliada` (el paquete `pipeline_sbs` no se modifica). Una fecha inexistente, como el 31 de febrero, devuelve `None`.

In [ ]:
# Reto de ampliación (opcional, sin nota)
MESES = {"enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6, "julio": 7, "agosto": 8,
         "setiembre": 9, "septiembre": 9, "octubre": 10, "noviembre": 11, "diciembre": 12}

CASOS_MONTOS = [("S/ 1 250,00", Decimal("1250.00")), ("1,250.50", Decimal("1250.50")), ("1.250,5", Decimal("1250.5"))]
CASOS_FECHAS = [("6 de julio de 2026", date(2026, 7, 6)), ("31 de febrero de 2026", None)]


def parsear_fecha_ampliada(texto):
    """Fecha (date) con formato numérico o con el mes escrito en texto; None si no es válida."""
    fecha = parsear_fecha(texto)
    if fecha is not None:
        return fecha
    partes = re.fullmatch(r"(\d{1,2}) de ([a-záéíóúñ]+) de (\d{4})", normalizar_texto(texto).lower())
    if partes is None or partes.group(2) not in MESES:
        return None
    try:
        return date(int(partes.group(3)), MESES[partes.group(2)], int(partes.group(1)))
    except ValueError:
        return None


# Reto extra
if parsear_fecha_ampliada("6 de julio de 2026") is not None:   # tras completar el TODO se ejecuta de nuevo esta celda
    for entrada, esperado in CASOS_MONTOS:
        assert parsear_monto(entrada) == esperado, f"El monto {entrada!r} debe dar {esperado}."
    for entrada, esperado in CASOS_FECHAS:
        assert parsear_fecha_ampliada(entrada) == esperado, f"La fecha {entrada!r} debe dar {esperado}."
    assert parsear_fecha_ampliada("11/08/2026") == date(2026, 8, 11), "Los formatos numéricos deben seguir funcionando."
    print("✔ Reto extra superado: 3 montos y 2 fechas normalizados")


### Guardar (≈1 min)

El participante pulsa `Ctrl+S` y ejecuta la celda de abajo: crea el commit en la VM y lo sube a `mio`, su repositorio personal (nunca a `origin`). **Debe verse:** `✔ Commit creado`, `✔ Subido a mio/trabajo-s02` y el último commit arriba de la lista. En github.com, el repositorio muestra el commit nuevo con ese mensaje.

In [ ]:
# Guardar el ejercicio: commit y subida a `mio`
guardar_avance("S02: E3 normalizacion")

## Ejercicio 4 (10 min) — Ingesta · M2.1, M2.4: Errores tipificados, cuarentena, duplicados y fichas

**Contexto:** la Mesa de Partes de *Financiera Altiplano DEMO S.A.* no puede detener la carga porque un archivo venga dañado. Cada fallo debe quedar con un motivo claro, el archivo debe copiarse a una cuarentena y un archivo repetido no debe procesarse dos veces.

**Insumo:** un lote de prueba que se genera en la primera celda a partir de `03_Estados_Financieros/EFI-0002.pdf` y `08_Vouchers/VOU-0002.jpg`: un PDF correcto y seis variantes defectuosas, cada una con un modo de fallo distinto; y el texto de referencia de `08_Vouchers/VOU-0001.jpg` para las fichas.

**Tarea:** el participante completa `verificar_apertura` (`calcular_sha256` y `enviar_a_cuarentena` vienen escritas) y el bucle que procesa los 7 archivos del lote de prueba sin detenerse. Después completa la ficha comparativa de PaddleOCR, LayoutParser y Donut y el mapa de herramientas (parte 2). Al final, las funciones de E1 a E4 se exportan al paquete `pipeline_sbs`.

**En palabras simples:** los 7 archivos son como siete envíos a la Mesa de Partes (un estado financiero correcto, uno vacío, uno cortado, una copia, un contrato dañado, una foto con nombre de PDF y un informe de auditoría con contraseña). Un error tipificado es un fallo con nombre propio (`archivo_vacio`, `pdf_cifrado`): el informe dice la causa exacta. La huella digital (hash SHA-256) se calcula con el contenido: dos archivos iguales por dentro dan la misma huella aunque se llamen distinto. La cuarentena es una carpeta aparte para lo que no se pudo leer. **Debe verse:** 7 registros, con 1 correcto, 5 fallos con 5 motivos distintos y 1 duplicado. **Si falla:** si el conteo no sale, se revisa qué archivo quedó sin explicar; si algo no sale, la «Ruta de apoyo (opcional)» de abajo trae el esqueleto.

> ◆ **Píldora P3: Lote o archivo individual (3 min).** *Pregunta de supervisión: si la Mesa de Partes envía 20 archivos y uno viene dañado, ¿se detiene todo el trabajo o se separa ese archivo y se sigue?* Un lote no se detiene por un archivo malo: se captura el error por archivo, se tipifica el motivo y se copia a cuarentena. El hash SHA-256 evita reprocesar duplicados, y `iterdir()` no garantiza el orden, por lo que se ordena.
> ```
> archivo → hash → ¿ya visto? ──sí──→ duplicado (no se reprocesa)
>                      └─no──→ formato por firma → abrir → ok
>                                                     └─ fallo → cuarentena + motivo
> ```

#### Ruta de apoyo (opcional)

El participante que prefiere un punto de partida sigue estos pasos. El esqueleto es parcial: los `...` son los huecos que el participante completa en las celdas de código del ejercicio.

1. `verificar_apertura` traduce cada fallo de la biblioteca al error tipificado correspondiente (`calcular_sha256` y `enviar_a_cuarentena` se entregan escritas).
2. En el bucle, el duplicado se detecta con `vistos` antes de abrir el archivo; el fallo se copia a cuarentena y se registra con su motivo.

```python
# dentro de verificar_apertura, rama PDF
try:
    doc = pymupdf.open(str(ruta))
except (pymupdf.FileDataError, pymupdf.EmptyFileError):
    raise ErrorPdfDanado(...)
if doc.needs_pass:
    raise ...

# en el bucle del lote
if sha in vistos:
    registros.append({...})
    continue
```

In [ ]:
# Errores tipificados de la ingesta (no se modifican)
class ErrorIngesta(Exception):
    """Base de los errores de ingesta. `motivo` es el código estable que se registra."""

    motivo = "error_inesperado"

    def __init__(self, detalle: str = "") -> None:
        super().__init__(detalle or self.motivo)
        self.detalle = detalle


class ErrorArchivoVacio(ErrorIngesta):
    """El archivo tiene 0 bytes."""

    motivo = "archivo_vacio"


class ErrorFormatoNoSoportado(ErrorIngesta):
    """El contenido no corresponde a ningún formato que el pipeline procese."""

    motivo = "formato_no_soportado"


class ErrorExtensionFalsa(ErrorIngesta):
    """La extensión declara un formato y la firma del contenido indica otro."""

    motivo = "extension_falsa"


class ErrorPdfCifrado(ErrorIngesta):
    """El PDF pide contraseña."""

    motivo = "pdf_cifrado"


class ErrorPdfDanado(ErrorIngesta):
    """El PDF está truncado o dañado (o solo se abre tras una reparación automática)."""

    motivo = "pdf_danado"


class ErrorDocxCorrupto(ErrorIngesta):
    """El .docx no es un ZIP válido o no contiene un documento Word."""

    motivo = "docx_corrupto"


class ErrorImagenInvalida(ErrorIngesta):
    """La imagen no se puede identificar."""

    motivo = "imagen_invalida"


class ErrorArchivoInaccesible(ErrorIngesta):
    """El sistema operativo no dejó leer el archivo."""

    motivo = "archivo_inaccesible"


def traducir_excepcion(error: BaseException) -> ErrorIngesta:
    """Convierte una excepción de biblioteca en un error tipificado."""
    if isinstance(error, ErrorIngesta):
        return error
    nombre = type(error).__name__
    detalle = f"{nombre}: {error}"
    if isinstance(error, zipfile.BadZipFile) or nombre in ("BadZipFile", "PackageNotFoundError"):
        return ErrorDocxCorrupto(detalle)
    if nombre in ("UnidentifiedImageError", "DecompressionBombError"):
        return ErrorImagenInvalida(detalle)
    if nombre in ("FileNotDecryptedError", "WrongPasswordError"):
        return ErrorPdfCifrado(detalle)
    if nombre in ("FileDataError", "EmptyFileError", "PdfReadError", "PdfStreamError", "PdfminerException"):
        return ErrorPdfDanado(detalle)
    if isinstance(error, OSError):
        return ErrorArchivoInaccesible(detalle)
    return ErrorIngesta(detalle)

In [ ]:
# Formato por firma, validación y despacho de extractores (no se modifica)
EXTENSIONES_POR_FORMATO = {
    "pdf": (".pdf",), "png": (".png",), "jpg": (".jpg", ".jpeg"), "tif": (".tif", ".tiff"),
    "docx": (".docx",), "eml": (".eml",), "html": (".html", ".htm"),
}


def detectar_formato(ruta):
    """Detecta el formato por la firma del contenido, no por la extensión."""
    ruta = Path(ruta)
    if ruta.stat().st_size == 0:
        return "vacio"
    with ruta.open("rb") as archivo:
        cabecera = archivo.read(2048)
    if b"%PDF-" in cabecera[:1024]:
        return "pdf"
    if cabecera.startswith(b"\x89PNG\r\n\x1a\n"):
        return "png"
    if cabecera.startswith(b"\xff\xd8\xff"):
        return "jpg"
    if cabecera.startswith((b"II*\x00", b"MM\x00*")):
        return "tif"
    if cabecera.startswith(b"PK\x03\x04"):
        return "docx" if ruta.suffix.lower() == ".docx" else "zip"
    texto = cabecera.decode("utf-8", errors="ignore").lstrip("﻿ \t\r\n").lower()
    if texto.startswith("<!doctype html") or "<html" in texto[:512]:
        return "html"
    cabeceras = re.findall(r"^(from|to|subject|date|mime-version|content-type|message-id):", texto, re.MULTILINE)
    if re.match(r"[a-z][\w-]*:", texto) and len(set(cabeceras)) >= 2:
        return "eml"
    return "desconocido"


def validar_formato_y_extension(ruta, formato):
    """Rechaza archivos vacíos, de formato desconocido o con extensión falsa."""
    ruta = Path(ruta)
    if formato == "vacio":
        raise ErrorArchivoVacio("el archivo tiene 0 bytes")
    if formato not in EXTENSIONES_POR_FORMATO:
        raise ErrorFormatoNoSoportado(f"formato '{formato}' no soportado")
    if ruta.suffix.lower() not in EXTENSIONES_POR_FORMATO[formato]:
        raise ErrorExtensionFalsa(f"la extensión '{ruta.suffix.lower()}' no corresponde al contenido ({formato})")


def extraer_texto(ruta, formato):
    """Elige el extractor de E1 según el formato detectado."""
    if formato == "pdf":
        return extraer_pdf_pymupdf(ruta)
    if formato == "docx":
        return extraer_docx(ruta)
    if formato == "eml":
        return extraer_eml(ruta)
    if formato == "html":
        return extraer_html(ruta)
    if formato in ("png", "jpg", "tif"):
        return extraer_imagen(ruta)
    raise ErrorFormatoNoSoportado(f"no hay extractor para el formato '{formato}'")

In [ ]:
def calcular_sha256(ruta):
    """Huella SHA-256 del contenido de un archivo, leído por bloques."""
    resumen = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            resumen.update(bloque)
    return resumen.hexdigest()


def verificar_apertura(ruta, formato):
    """Comprueba que el archivo se puede abrir y lanza el error tipificado que corresponde."""
    ruta = Path(ruta)
    if formato == "pdf":
        try:
            doc = pymupdf.open(str(ruta))
        except (pymupdf.FileDataError, pymupdf.EmptyFileError) as error:
            raise ErrorPdfDanado(f"{type(error).__name__}: no se pudo abrir el PDF") from error
        with doc:
            if doc.needs_pass:
                raise ErrorPdfCifrado("el PDF pide contraseña")
            if doc.is_repaired:
                raise ErrorPdfDanado("el PDF estaba dañado y solo se abrió tras una reparación automática")
    elif formato == "docx":
        if not zipfile.is_zipfile(ruta):
            raise ErrorDocxCorrupto("el archivo no es un ZIP válido")


def enviar_a_cuarentena(ruta, cuarentena, motivo, detalle=""):
    """Copia (no mueve) un archivo defectuoso a la cuarentena y deja junto a él su motivo."""
    ruta, cuarentena = Path(ruta), Path(cuarentena)
    cuarentena.mkdir(parents=True, exist_ok=True)
    destino = cuarentena / ruta.name
    temporal = destino.with_name(destino.name + ".tmp")
    shutil.copy2(ruta, temporal)
    temporal.replace(destino)
    nota = {"archivo": ruta.name, "motivo": motivo, "detalle": detalle[:300]}
    destino.with_name(destino.name + ".motivo.json").write_text(json.dumps(nota, ensure_ascii=False, indent=2), encoding="utf-8")
    return destino

In [ ]:
# Generación del lote de prueba (no se modifica): un PDF correcto y 6 variantes defectuosas, a partir de documentos de práctica
LOTE_E4 = Path(tempfile.mkdtemp(prefix="lote_e4_"))
origen_pdf = DATOS / "03_Estados_Financieros" / "EFI-0002.pdf"
origen_voucher = DATOS / "08_Vouchers" / "VOU-0002.jpg"
contenido_pdf = origen_pdf.read_bytes()

shutil.copy2(origen_pdf, LOTE_E4 / "E4-01_balance.pdf")                                       # correcto
(LOTE_E4 / "E4-02_vacio.pdf").write_bytes(b"")                                                # 0 bytes
(LOTE_E4 / "E4-03_truncado.pdf").write_bytes(contenido_pdf[: len(contenido_pdf) // 2])        # copia cortada a la mitad
shutil.copy2(origen_pdf, LOTE_E4 / "E4-04_balance_copia.pdf")                                 # copia exacta del primero
(LOTE_E4 / "E4-05_contrato.docx").write_bytes(b"PK\x03\x04" + contenido_pdf[200:700])              # ZIP roto con extensión .docx
shutil.copy2(origen_voucher, LOTE_E4 / "E4-06_voucher.pdf")                                   # imagen con extensión .pdf
with pymupdf.open(str(origen_pdf)) as doc:                                                    # PDF con contraseña
    doc.save(str(LOTE_E4 / "E4-07_reservado.pdf"), encryption=pymupdf.PDF_ENCRYPT_AES_256, user_pw="clave", owner_pw="duena")

CUARENTENA = Path(tempfile.mkdtemp(prefix="cuarentena_"))
ARCHIVOS_E4 = sorted(p.name for p in LOTE_E4.iterdir())
print(len(ARCHIVOS_E4), "archivos en el lote de prueba:", ARCHIVOS_E4)

In [ ]:
vistos = {}        # huella -> nombre del primer archivo con ese contenido
aperturas = []     # archivos que se abrieron de verdad
registros = []
for nombre in ARCHIVOS_E4:
    ruta = LOTE_E4 / nombre
    sha = calcular_sha256(ruta)
    if sha in vistos:
        registros.append({"archivo": nombre, "estado": "duplicado", "motivo": "duplicado", "duplicado_de": vistos[sha]})
        continue
    try:
        formato = detectar_formato(ruta)
        validar_formato_y_extension(ruta, formato)
        aperturas.append(nombre)
        verificar_apertura(ruta, formato)
        vistos[sha] = nombre
        registros.append({"archivo": nombre, "estado": "ok", "motivo": "", "duplicado_de": ""})
    except ErrorIngesta as error:
        enviar_a_cuarentena(ruta, CUARENTENA, error.motivo, str(error))
        registros.append({"archivo": nombre, "estado": "fallo", "motivo": error.motivo, "duplicado_de": ""})

tabla_e4 = pd.DataFrame(registros)
print(tabla_e4.to_string(index=False))

In [ ]:
# ✔ Checkpoint 4
esperado = {"E4-02_vacio.pdf": "archivo_vacio", "E4-03_truncado.pdf": "pdf_danado", "E4-05_contrato.docx": "docx_corrupto",
            "E4-06_voucher.pdf": "extension_falsa", "E4-07_reservado.pdf": "pdf_cifrado"}
assert len(tabla_e4) == 7, "Debe haber un registro por archivo: el lote no se detuvo."
motivos = {r["archivo"]: r["motivo"] for r in registros if r["estado"] == "fallo"}
assert motivos == esperado, f"Motivos obtenidos: {motivos}"
assert len(set(motivos.values())) == 5, "Cada fallo debe tener un motivo distinto."
duplicados = [r for r in registros if r["estado"] == "duplicado"]
assert len(duplicados) == 1 and duplicados[0]["archivo"] == "E4-04_balance_copia.pdf"     and duplicados[0]["duplicado_de"] == "E4-01_balance.pdf", f"Duplicados detectados: {duplicados}"
assert "E4-04_balance_copia.pdf" not in aperturas, "El duplicado no debe abrirse ni reprocesarse."
copias = sorted(p.name for p in CUARENTENA.iterdir() if not p.name.endswith(".json"))
assert copias == sorted(esperado), f"En cuarentena hay: {copias}"
assert all((LOTE_E4 / nombre).exists() for nombre in ARCHIVOS_E4), "La cuarentena copia, no mueve."
assert json.loads((CUARENTENA / "E4-07_reservado.pdf.motivo.json").read_text(encoding="utf-8"))["motivo"] == "pdf_cifrado"
enviar_a_cuarentena(LOTE_E4 / "E4-07_reservado.pdf", CUARENTENA, "pdf_cifrado")      # repetir la copia no debe fallar
print("✔ Checkpoint 4 superado: 5 fallos con motivos distintos en cuarentena y 1 duplicado sin reprocesar")

### E4, parte 2 (M2.4): Fichas comparativas de PaddleOCR, LayoutParser y Donut y mapa de herramientas

**Contexto:** antes de decidir con qué herramienta se leerá un voucher escaneado, el equipo de *Financiera Altiplano DEMO S.A.* compara tres opciones en un cuadro, como se comparan tres cotizaciones antes de comprar. PaddleOCR se ejecuta en la sesión 3; hoy el participante lee un **texto de referencia**, no el resultado de una ejecución.

**Insumo:** el texto de referencia de la celda siguiente (lo que un OCR reconocería en `08_Vouchers/VOU-0001.jpg`, una línea por elemento; no se ejecutó PaddleOCR) y la tabla de abajo.

**Tarea:** el participante (1) lee el texto de referencia: cuenta sus líneas y obtiene el monto del depósito con `parsear_monto`; (2) completa los dos primeros campos de la ficha comparativa de las tres herramientas con frases propias, a partir de la tabla (el tercer campo viene escrito); (3) arma el mapa de herramientas por tipo de tarea.

| Herramienta | Qué hace | Qué necesita o limita | Cuándo se ejecuta |
|---|---|---|---|
| PaddleOCR 3.x (Apache-2.0) | Lee las líneas de texto de una imagen o de un PDF escaneado | Usa dos modelos (detección y reconocimiento); los modelos *mobile* son ligeros y corren en CPU | Sesión 3, en la VM |
| LayoutParser 0.3.4 | Detecta regiones (título, tabla, firma) con modelos de detección de objetos | Última versión de 2022; sus modelos necesitan Detectron2, que se compila y no tiene soporte oficial en Windows | Solo se analiza: no se ejecuta en el curso |
| Donut (`donut-base-finetuned-cord-v2`, licencia MIT) | Convierte la imagen de un documento directamente en JSON, sin OCR previo | Se ejecuta con Transformers; se preentrenó en inglés, chino, japonés y coreano, sin español | Se retoma en la sesión 4, con Transformers |

**En palabras simples:** PaddleOCR es el programa que lee las letras de una foto, como transcribir a mano una constancia de depósito; LayoutParser señala dónde está cada parte de la hoja y Donut intenta entregar la hoja ya convertida en datos. La ficha es el cuadro comparativo que se arma antes de decidir. **Debe verse:** `✔ Checkpoint 5 superado` con el monto leído del texto de referencia y la ficha de las tres herramientas completa. **Si falla:** un `AssertionError` indica qué campo de la ficha tiene menos de 4 palabras o qué herramienta del mapa no corresponde a su tarea; no es una falla del participante.

Fuentes: https://pypi.org/project/layoutparser/ ; https://huggingface.co/naver-clova-ix/donut-base-finetuned-cord-v2 ; https://github.com/PaddlePaddle/PaddleOCR

In [ ]:
# TEXTO DE REFERENCIA (no se modifica; no es el resultado de ejecutar PaddleOCR): lo que un OCR reconocería en
# `08_Vouchers/VOU-0001.jpg`, una línea por elemento
SALIDA_REFERENCIA_PADDLE = [
    "Banco Costa Norte DEMO S.A.",
    "CONSTANCIA DE DEPOSITO",
    "Operacion:",
    "35830690",
    "Fecha:",
    "23/06/2026",
    "Hora:",
    "13:10",
    "Cuenta:",
    "999-6919903748",
    "Titular:",
    "Paola Benavides Soto",
    "Depositante:",
    "Miguel Fuentes Vargas",
    "DNI:",
    "99924030",
    "MONTO S/ 16,798.00"
]

print("Texto de referencia (PaddleOCR no se ejecutó):", len(SALIDA_REFERENCIA_PADDLE), "líneas")
print("\n".join(SALIDA_REFERENCIA_PADDLE))


In [ ]:
n_lineas = len(SALIDA_REFERENCIA_PADDLE)
linea_monto = next(linea for linea in SALIDA_REFERENCIA_PADDLE if "MONTO" in linea)
monto_leido = parsear_monto(linea_monto.replace("MONTO", ""))

ficha_comparativa = {
    "PaddleOCR": {
        "para_que_sirve": "Lee las líneas de texto de una imagen o de un PDF escaneado",
        "que_necesita_o_limita": "Usa un modelo de detección y otro de reconocimiento, ligeros en su versión mobile",
        "cuando_se_ejecuta": "En la sesión 3, dentro de la VM del curso",
    },
    "LayoutParser": {
        "para_que_sirve": "Detecta regiones del documento como título, tabla o firma",
        "que_necesita_o_limita": "Sus modelos exigen Detectron2, que se compila y no tiene soporte oficial en Windows",
        "cuando_se_ejecuta": "Solo se analiza en la ficha y no se ejecuta en el curso",
    },
    "Donut": {
        "para_que_sirve": "Convierte la imagen de un documento directamente en JSON sin OCR previo",
        "que_necesita_o_limita": "Exige Transformers y se preentrenó sin español, solo con inglés, chino, japonés y coreano",
        "cuando_se_ejecuta": "Se retoma en la sesión 4 junto con Transformers",
    },
}

mapa_herramientas = {
    "texto de un PDF con capa de texto": "pymupdf",
    "tabla en un PDF con texto": "tabula-py",
    "contenido de un DOCX": "python-docx",
    "cuerpo y adjuntos de un correo EML": "email",
    "texto de una página HTML": "beautifulsoup",
    "texto de una imagen o PDF escaneado": "paddleocr",
    "regiones de un documento (título, tabla, firma)": "layoutparser",
    "documento convertido directo a JSON sin OCR": "donut",
}

In [ ]:
# ✔ Checkpoint 5 (ficha completa)
assert n_lineas == len(SALIDA_REFERENCIA_PADDLE) and n_lineas > 5, "n_lineas debe ser la cantidad de líneas del texto de referencia."
assert "MONTO" in linea_monto, "linea_monto debe ser la línea del texto de referencia que contiene MONTO."
assert monto_leido == Decimal("16798.00"), f"El monto del texto de referencia es S/ 16 798,00 y se leyó {monto_leido}."
assert any("DEPOSITO" in linea.upper() for linea in SALIDA_REFERENCIA_PADDLE), "El texto de referencia trae 'CONSTANCIA DE DEPOSITO'."
assert set(ficha_comparativa) == {"PaddleOCR", "LayoutParser", "Donut"}, "La ficha debe cubrir las tres herramientas."
for herramienta, ficha in ficha_comparativa.items():
    for campo, frase in ficha.items():
        assert len(frase.split()) >= 4, f"Falta completar '{campo}' de {herramienta} con una frase de al menos 4 palabras."
permitidas = {
    "texto de un PDF con capa de texto": {"pypdf", "pdfplumber", "pymupdf"},
    "tabla en un PDF con texto": {"tabula-py", "camelot", "pdfplumber", "pymupdf"},
    "contenido de un DOCX": {"python-docx"},
    "cuerpo y adjuntos de un correo EML": {"email"},
    "texto de una página HTML": {"beautifulsoup"},
    "texto de una imagen o PDF escaneado": {"paddleocr", "tesseract"},
    "regiones de un documento (título, tabla, firma)": {"layoutparser"},
    "documento convertido directo a JSON sin OCR": {"donut"},
}
for tarea, herramienta in mapa_herramientas.items():
    assert herramienta.lower() in permitidas[tarea], f"La herramienta de '{tarea}' requiere revisión: {herramienta!r}"
print(f"✔ Checkpoint 5 superado: monto leído del texto de referencia S/ {monto_leido}, ficha de las tres herramientas completa y mapa coherente")


### Exportación de las funciones de E1 a E4 a `pipeline_sbs`

Los lectores, las funciones de normalización, los errores tipificados y las funciones de ingesta del participante pasan a módulos de su paquete. Después se comprueba que `pytest` sigue en verde. En palabras simples: exportar es guardar las funciones recién escritas en la caja de herramientas del proyecto del equipo, y `pytest` corre las pruebas automáticas, como una lista de verificación. **Debe verse:** líneas `✔ ... exportada a ...` y las pruebas en verde. **Si falla:** un mensaje en rojo no significa que se perdió el trabajo: se vuelve a ejecutar la celda de exportación.

In [ ]:
# Exportación de las funciones de E1 a E4 a pipeline_sbs (celda estándar: no se modifica)
import ast
import inspect


def codigo_fuente(objeto):
    """Código fuente de una función o clase definida en este notebook."""
    try:
        return inspect.getsource(objeto)
    except (OSError, TypeError):
        for celda in reversed(get_ipython().user_ns.get("In", [])):
            if f"class {objeto.__name__}" not in celda:
                continue
            for nodo in ast.parse(celda).body:
                if isinstance(nodo, ast.ClassDef) and nodo.name == objeto.__name__:
                    inicio = nodo.decorator_list[0].lineno if nodo.decorator_list else nodo.lineno
                    return "\n".join(celda.splitlines()[inicio - 1:nodo.end_lineno]) + "\n"
        raise


ENCABEZADOS = {
    "ingesta/errores.py": '''"""Errores tipificados de la ingesta: cada modo de fallo tiene su clase y un motivo estable."""
from __future__ import annotations

import zipfile
''',
    "ingesta/extractores.py": '''"""Extractores de texto por formato: PDF (tres bibliotecas), DOCX, correo EML, HTML e imagen."""
from __future__ import annotations

import re
from dataclasses import dataclass, field
from email import policy
from email.parser import BytesParser
from pathlib import Path

import pdfplumber
import pymupdf
import pypdf
from bs4 import BeautifulSoup
from docx import Document
from docx.table import Table
from docx.text.paragraph import Paragraph
from PIL import Image

from pipeline_sbs.ingesta.errores import ErrorFormatoNoSoportado

pymupdf.TOOLS.mupdf_display_errors(False)
''',
    "preprocesamiento/normalizacion.py": '''"""Normalización de texto extraído: Unicode, espacios, guiones de corte, fechas y montos en soles."""
from __future__ import annotations

import re
import unicodedata
from datetime import date, datetime
from decimal import Decimal

ESPACIOS_ESPECIALES = "\\u00a0\\u1680\\u2000\\u2001\\u2002\\u2003\\u2004\\u2005\\u2006\\u2007\\u2008\\u2009\\u200a\\u202f\\u205f\\u3000"
INVISIBLES = "\\u00ad\\u200b\\u200c\\u200d\\ufeff"          # guion blando, ancho cero, marca de orden de bytes
LIGADURAS = {"\\ufb00": "ff", "\\ufb01": "fi", "\\ufb02": "fl", "\\ufb03": "ffi", "\\ufb04": "ffl"}
FORMATOS_FECHA = ("%d/%m/%Y", "%d-%m-%Y", "%Y-%m-%d")
''',
    "ingesta/cargador.py": '''"""Cargador de lotes: abre cada archivo según su contenido, tipifica los fallos y calcula estadísticas."""
from __future__ import annotations

import hashlib
import json
import logging
import re
import shutil
import statistics
import time
import zipfile
from dataclasses import asdict, dataclass, field
from pathlib import Path

import pandas as pd
import pymupdf

from pipeline_sbs.ingesta.errores import (
    ErrorArchivoVacio,
    ErrorDocxCorrupto,
    ErrorExtensionFalsa,
    ErrorFormatoNoSoportado,
    ErrorIngesta,
    ErrorPdfCifrado,
    ErrorPdfDanado,
    traducir_excepcion,
)
from pipeline_sbs.ingesta.extractores import extraer_texto
from pipeline_sbs.preprocesamiento.normalizacion import normalizar_texto

registro = logging.getLogger(__name__)

EXTENSIONES_POR_FORMATO = {
    "pdf": (".pdf",), "png": (".png",), "jpg": (".jpg", ".jpeg"), "tif": (".tif", ".tiff"),
    "docx": (".docx",), "eml": (".eml",), "html": (".html", ".htm"),
}
TIPO_POR_EXTENSION = {
    ".pdf": "pdf", ".png": "imagen", ".jpg": "imagen", ".jpeg": "imagen", ".tif": "imagen", ".tiff": "imagen",
    ".docx": "word", ".eml": "correo", ".html": "html", ".htm": "html",
}
FORMATOS_IMAGEN = ("png", "jpg", "tif")
''',
}

EXPORTACIONES = {
    "ingesta/errores.py": ([ErrorIngesta, ErrorArchivoVacio, ErrorFormatoNoSoportado, ErrorExtensionFalsa, ErrorPdfCifrado,
                            ErrorPdfDanado, ErrorDocxCorrupto, ErrorImagenInvalida, ErrorArchivoInaccesible,
                            traducir_excepcion]),
    "ingesta/extractores.py": ([Extraccion, extraccion_desde_paginas, extraer_pdf_pypdf, extraer_pdf_pdfplumber,
                                extraer_pdf_pymupdf, filas_de_tabla, extraer_docx, extraer_eml, decodificar_html, extraer_html,
                                extraer_imagen, extraer_texto]),
    "preprocesamiento/normalizacion.py": ([normalizar_texto, unir_guiones, parsear_fecha, parsear_monto]),
    "ingesta/cargador.py": ([calcular_sha256, detectar_formato, validar_formato_y_extension, verificar_apertura,
                             enviar_a_cuarentena]),
}

raiz_pipeline = localizar_paquete()
if raiz_pipeline is None:
    print("No se encuentra el paquete clonado ni el respaldo; se usa el paquete ya instalado.")
else:
    for modulo, objetos in EXPORTACIONES.items():
        destino = raiz_pipeline / "pipeline_sbs" / modulo
        destino.parent.mkdir(parents=True, exist_ok=True)
        actual = destino.read_text(encoding="utf-8") if destino.exists() else ENCABEZADOS[modulo]
        for objeto in objetos:
            marca = f"class {objeto.__name__}" if inspect.isclass(objeto) else f"def {objeto.__name__}("
            if marca in actual:
                print(f"{objeto.__name__} ya está en {modulo}; no se duplica. Si la función se corrigió, se borra su versión anterior del módulo y se exporta de nuevo.")
                continue
            actual = actual.rstrip("\n") + "\n\n\n" + codigo_fuente(objeto)
            print(f"✔ {objeto.__name__} exportada a {modulo}")
        destino.write_text(actual.rstrip("\n") + "\n", encoding="utf-8")

In [ ]:
# ✔ Checkpoint de la exportación (E1 a E4)
import importlib
import subprocess

importlib.invalidate_caches()
for nombre_modulo, nombres in {
    "pipeline_sbs.ingesta.errores": ["ErrorIngesta", "ErrorPdfCifrado", "traducir_excepcion"],
    "pipeline_sbs.ingesta.extractores": ["extraer_pdf_pypdf", "extraer_pdf_pdfplumber", "extraer_pdf_pymupdf", "filas_de_tabla",
                                         "extraer_docx", "extraer_eml", "extraer_html", "extraer_imagen", "extraer_texto"],
    "pipeline_sbs.preprocesamiento.normalizacion": ["normalizar_texto", "unir_guiones", "parsear_fecha", "parsear_monto"],
    "pipeline_sbs.ingesta.cargador": ["calcular_sha256", "detectar_formato", "validar_formato_y_extension",
                                      "verificar_apertura", "enviar_a_cuarentena"],
}.items():
    modulo = importlib.reload(importlib.import_module(nombre_modulo))
    for nombre in nombres:
        assert hasattr(modulo, nombre), f"Falta {nombre} en {nombre_modulo}: falta ejecutar la celda de exportación."

carpeta_pruebas = Path(pipeline_sbs.__file__).parent / "tests"
entorno = entorno_pruebas()
corrida = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(carpeta_pruebas)],
    capture_output=True, text=True, env=entorno, timeout=300,
)
print(corrida.stdout[-400:])
assert corrida.returncode == 0, "pytest falla: el mensaje de arriba detalla el error."
print("✔ Exportación superada: el pipeline_sbs tiene los módulos de E1 a E4 y las pruebas pasan")

### Guardar (≈1 min)

El participante pulsa `Ctrl+S` y ejecuta la celda de abajo: crea el commit en la VM y lo sube a `mio`, su repositorio personal (nunca a `origin`). **Debe verse:** `✔ Commit creado`, `✔ Subido a mio/trabajo-s02` y el último commit arriba de la lista. En github.com, el repositorio muestra el commit nuevo con ese mensaje.

In [ ]:
# Guardar el ejercicio: commit y subida a `mio`
guardar_avance("S02: E4 errores, cuarentena y fichas")

## Ejercicio 5 (16 min) — Integra todo · M2.5: La clase de carga (`CargadorDocumentos`)

**Contexto:** la Mesa de Partes de *Financiera Altiplano DEMO S.A.* ya tiene las piezas sueltas (huella, formato por firma, validación, extracción, normalización y cuarentena). Ahora necesita una clase de carga que procese un lote de 20 documentos mixtos, un registro por archivo, y que entregue las estadísticas de la extracción, como el informe del día de la Mesa de Partes.

**Insumo:** los 20 archivos de `02_Datos_Practica/01_Lote_S02/` y las funciones que el participante escribió en E1 a E4 (ya exportadas al paquete).

**Tarea:** (1) ejecutar la celda de piezas escritas (`ResultadoCarga`, `tabla_resultados` y `estadisticas`); (2) completar `cargar_uno` de `CargadorDocumentos` con las funciones de E1 a E4, en este orden: huella y formato, validación y apertura, duplicado, extracción con normalización y decisión de OCR, y cuarentena ante un fallo; (3) correr `cargar_lote()` sobre los 20 archivos y obtener `estadisticas()`; (4) pasar el checkpoint C6; (5) exportar la clase al paquete.

**En palabras simples:** una clase es un molde que junta datos (la carpeta, la cuarentena, las huellas ya vistas) y las acciones que trabajan con ellos (`cargar_uno`, `cargar_lote`). `cargar_uno` procesa un archivo y nunca se cae por su contenido; `cargar_lote` lo repite con cada archivo y empieza sin memoria, por eso repetirlo da lo mismo. **Debe verse:** `✔ Checkpoint 6 (C6) superado` con 20 registros: 14 ok, 5 fallos y 1 duplicado, y los 5 escaneados ok con `requiere_ocr`. **Si falla:** el `AssertionError` de C6 indica el tipo de archivo cuyos conteos no coinciden; una celda de `cargar_uno` sin completar deja el mensaje «Falta completar cargar_uno». Si algo no sale, la «Ruta de apoyo (opcional)» de abajo trae el esqueleto.

#### Ruta de apoyo (opcional)

El participante que prefiere un punto de partida sigue estos pasos dentro de `cargar_uno`; los `...` son los huecos.

1. `sha = calcular_sha256(ruta)` y `formato = detectar_formato(ruta)`.
2. `validar_formato_y_extension(ruta, formato)` y `verificar_apertura(ruta, formato)`: lanzan el error tipificado que corresponde.
3. Si `sha` ya está en `self._vistos`, el resultado es `ResultadoCarga(..., estado="duplicado", duplicado_de=...)` y no se extrae nada.
4. Si no: `extraer_texto`, `normalizar_texto`, `requiere_ocr` (imagen, o PDF con menos de `self.umbral_ocr` caracteres por página) y `self._vistos[sha] = ruta.name`.
5. En el `except`: `traducir_excepcion`, resultado con estado `"fallo"` y `enviar_a_cuarentena(...)`.

```python
resultado = ResultadoCarga(archivo=ruta.name, tipo=tipo, formato=formato, estado="fallo",
                           motivo=tipificado.motivo, detalle=detalle, sha256=sha)
```

In [ ]:
# Piezas escritas de E5 (no se modifica): resultado por archivo, tipos, tabla y estadísticas
FORMATOS_IMAGEN = ("png", "jpg", "tif")
TIPO_POR_EXTENSION = {".pdf": "pdf", ".png": "imagen", ".jpg": "imagen", ".jpeg": "imagen", ".tif": "imagen",
                      ".tiff": "imagen", ".docx": "word", ".eml": "correo", ".html": "html", ".htm": "html"}


@dataclass
class ResultadoCarga:
    """Registro de un archivo procesado (estado: ok, fallo o duplicado)."""

    archivo: str
    tipo: str
    formato: str
    estado: str
    motivo: str = ""
    detalle: str = ""
    paginas: int = 0
    caracteres: int = 0
    segundos: float = 0.0
    requiere_ocr: bool = False
    sha256: str = ""
    duplicado_de: str = ""
    texto: str = field(default="", repr=False)

    def como_dict(self) -> dict:
        """Devuelve el registro sin el texto, listo para una tabla."""
        datos = asdict(self)
        datos.pop("texto")
        return datos


def tabla_resultados(resultados: list) -> pd.DataFrame:
    """Tabla con un registro por archivo (sin el texto)."""
    return pd.DataFrame([r.como_dict() for r in resultados])


def estadisticas(resultados: list) -> pd.DataFrame:
    """Resumen del lote por tipo de archivo, con una fila final TOTAL.

    El porcentaje que requiere OCR se calcula solo sobre los archivos ok.
    """

    def resumir(grupo: pd.DataFrame) -> dict:
        n = len(grupo)
        ok = int((grupo["estado"] == "ok").sum())
        fallo = int((grupo["estado"] == "fallo").sum())
        duplicado = int((grupo["estado"] == "duplicado").sum())
        con_ocr = int(((grupo["estado"] == "ok") & grupo["requiere_ocr"]).sum())
        return {
            "n": n, "ok": ok, "fallo": fallo, "duplicado": duplicado,
            "pct_ok": round(100 * ok / n, 1), "pct_fallo": round(100 * fallo / n, 1),
            "pct_duplicado": round(100 * duplicado / n, 1),
            "paginas": int(grupo["paginas"].sum()), "caracteres": int(grupo["caracteres"].sum()),
            "segundos_total": round(float(grupo["segundos"].sum()), 6),
            "segundos_mediana": round(float(grupo["segundos"].median()), 6),
            "requiere_ocr": con_ocr, "pct_requiere_ocr": round(100 * con_ocr / ok, 1) if ok else 0.0,
        }

    tabla = tabla_resultados(resultados)
    filas = {tipo: resumir(grupo) for tipo, grupo in tabla.groupby("tipo", sort=True)}
    filas["TOTAL"] = resumir(tabla)
    return pd.DataFrame.from_dict(filas, orient="index")

In [ ]:
class CargadorDocumentos:
    """Carga un lote de archivos mixtos sin que un archivo defectuoso detenga el proceso."""

    def __init__(self, carpeta, cuarentena, umbral_ocr: int = 20) -> None:
        self.carpeta = Path(carpeta)
        self.cuarentena = Path(cuarentena)
        self.umbral_ocr = umbral_ocr
        self._vistos: dict = {}      # huella -> nombre del primer archivo con ese contenido

    def cargar_uno(self, ruta) -> ResultadoCarga:
        """Procesa un archivo y devuelve su ResultadoCarga. Nunca lanza una excepción por el contenido del archivo."""
        ruta = Path(ruta)
        inicio = time.perf_counter()
        tipo = TIPO_POR_EXTENSION.get(ruta.suffix.lower(), "otro")
        sha, formato = "", "desconocido"
        resultado = None
        try:
            sha = calcular_sha256(ruta)
            formato = detectar_formato(ruta)
            validar_formato_y_extension(ruta, formato)
            verificar_apertura(ruta, formato)
            original = self._vistos.get(sha)
            if original is not None:
                resultado = ResultadoCarga(archivo=ruta.name, tipo=tipo, formato=formato, estado="duplicado",
                                           motivo="duplicado", detalle=f"mismo contenido que {original}",
                                           sha256=sha, duplicado_de=original)
            else:
                extraccion = extraer_texto(ruta, formato)
                texto = normalizar_texto(extraccion.texto)
                por_pagina = len(texto) / max(extraccion.paginas, 1)
                resultado = ResultadoCarga(
                    archivo=ruta.name, tipo=tipo, formato=formato, estado="ok", paginas=extraccion.paginas,
                    caracteres=len(texto), sha256=sha, texto=texto,
                    requiere_ocr=(formato in FORMATOS_IMAGEN or (formato == "pdf" and por_pagina < self.umbral_ocr)),
                )
                self._vistos[sha] = ruta.name
        except Exception as error:  # noqa: BLE001 - un archivo malo no debe detener el lote
            tipificado = error if isinstance(error, ErrorIngesta) else traducir_excepcion(error)
            detalle = (tipificado.detalle or str(tipificado)).replace(str(ruta), ruta.name)
            resultado = ResultadoCarga(archivo=ruta.name, tipo=tipo, formato=formato, estado="fallo",
                                       motivo=tipificado.motivo, detalle=detalle, sha256=sha)
            try:
                enviar_a_cuarentena(ruta, self.cuarentena, tipificado.motivo, detalle)
            except OSError:
                pass     # si la cuarentena tampoco se puede escribir, el lote igual continúa
        resultado.segundos = time.perf_counter() - inicio
        return resultado

    def cargar_lote(self) -> list:
        """Procesa todos los archivos de la carpeta, en orden alfabético, y devuelve un ResultadoCarga por archivo."""
        self._vistos = {}
        archivos = sorted(p for p in self.carpeta.iterdir() if p.is_file() and not p.name.startswith("."))
        return [self.cargar_uno(p) for p in archivos]

In [ ]:
cuarentena_e5 = Path(tempfile.mkdtemp(prefix="cuarentena_e5_"))
cargador = CargadorDocumentos(LOTE, cuarentena_e5)

resultados = cargador.cargar_lote()
resumen = estadisticas(resultados)

print(len(resultados), "registros")
print(resumen[["n", "ok", "fallo", "duplicado", "caracteres", "requiere_ocr", "pct_requiere_ocr"]].to_string())

In [ ]:
# ✔ Checkpoint 6 (C6): la clase de carga sobre los 20 archivos
ESPERADO_POR_TIPO = {      # tipo: (n, ok, fallo, duplicado, caracteres, requiere_ocr)
    "correo": (2, 2, 0, 0, 486, 0),
    "html": (2, 2, 0, 0, 1383, 0),
    "imagen": (4, 4, 0, 0, 0, 4),
    "pdf": (10, 5, 4, 1, 4284, 1),
    "word": (2, 1, 1, 0, 10357, 0),
    "TOTAL": (20, 14, 5, 1, 16510, 5),
}
MOTIVOS_ESPERADOS = {"LOT-0002.pdf": "extension_falsa", "LOT-0006.docx": "docx_corrupto", "LOT-0007.pdf": "pdf_cifrado",
                     "LOT-0014.pdf": "pdf_danado", "LOT-0016.pdf": "archivo_vacio"}

assert len(resultados) == 20 and resumen is not None, "Debe haber un registro por archivo (20) y la tabla `resumen`: se revisan cargar_lote() y estadisticas()."
for tipo, esperado in ESPERADO_POR_TIPO.items():
    obtenido = tuple(int(resumen.loc[tipo, columna]) for columna in ("n", "ok", "fallo", "duplicado", "caracteres", "requiere_ocr"))
    assert obtenido == esperado, \
        f"Tipo '{tipo}': se obtuvo {obtenido} y se esperaba {esperado} (n, ok, fallo, duplicado, caracteres, requiere_ocr)."

por_archivo = {r.archivo: r for r in resultados}
assert {a: por_archivo[a].motivo for a in MOTIVOS_ESPERADOS} == MOTIVOS_ESPERADOS, "Los 5 fallos deben llevar su motivo tipificado."
assert por_archivo["LOT-0019.pdf"].estado == "duplicado" and por_archivo["LOT-0019.pdf"].duplicado_de == "LOT-0001.pdf", "LOT-0019 es copia de LOT-0001."
escaneados = [r for r in resultados if r.tipo == "imagen"] + [por_archivo["LOT-0012.pdf"]]
assert all(r.estado == "ok" and r.requiere_ocr for r in escaneados), "Los escaneados quedan ok con requiere_ocr (sin error)."
assert resumen.loc["TOTAL", "pct_requiere_ocr"] == 35.7, "pct_requiere_ocr se calcula sobre los archivos ok (5 de 14)."

copias = sorted(p.name for p in cuarentena_e5.iterdir() if not p.name.endswith(".json"))
assert copias == sorted(MOTIVOS_ESPERADOS), f"En la cuarentena hay: {copias}"
inexistente = CargadorDocumentos(LOTE, cuarentena_e5).cargar_uno(LOTE / "LOT-9999.pdf")
assert inexistente.estado == "fallo", "Un archivo defectuoso (aquí, inexistente) no detiene el lote: cargar_uno devuelve estado fallo."

segunda = CargadorDocumentos(LOTE, cuarentena_e5).cargar_lote()
clave = ["archivo", "estado", "motivo", "formato", "paginas", "caracteres", "requiere_ocr", "sha256", "duplicado_de"]
assert tabla_resultados(resultados)[clave].equals(tabla_resultados(segunda)[clave]), "La segunda corrida debe ser idéntica."
assert max(r.segundos for r in resultados) < 5, "Cada archivo debe procesarse en menos de 5 segundos."
print("✔ Checkpoint 6 (C6) superado: 20 registros, 14 ok, 5 fallos y 1 duplicado; los 5 escaneados quedan ok con requiere_ocr")

In [ ]:
# Exportación de la clase de carga a pipeline_sbs (celda estándar: no se modifica)
PRUEBAS_CARGADOR = r'''"""Pruebas mínimas de la ingesta de S02 con archivos sintéticos."""
import pymupdf
import pytest
from PIL import Image

from pipeline_sbs.ingesta.cargador import (
    CargadorDocumentos,
    calcular_sha256,
    detectar_formato,
    enviar_a_cuarentena,
    validar_formato_y_extension,
    verificar_apertura,
)
from pipeline_sbs.ingesta.errores import ErrorArchivoVacio, ErrorExtensionFalsa, ErrorPdfCifrado
from pipeline_sbs.ingesta.extractores import extraer_pdf_pymupdf


def _pdf(ruta, texto="Balance DEMO", **opciones):
    doc = pymupdf.open()
    doc.new_page().insert_text((72, 72), texto)
    doc.save(str(ruta), **opciones)
    doc.close()
    return ruta


def _lote(tmp_path):
    lote = tmp_path / "lote"
    lote.mkdir()
    _pdf(lote / "a_balance.pdf")
    (lote / "b_copia.pdf").write_bytes((lote / "a_balance.pdf").read_bytes())
    (lote / "c_vacio.pdf").write_bytes(b"")
    Image.new("RGB", (20, 20), "white").save(lote / "d_voucher.png")
    return lote


def test_extractor_pymupdf_lee_el_texto(tmp_path):
    assert "Balance DEMO" in extraer_pdf_pymupdf(_pdf(tmp_path / "a.pdf")).texto


def test_huella_igual_para_copias_exactas(tmp_path):
    lote = _lote(tmp_path)
    assert calcular_sha256(lote / "a_balance.pdf") == calcular_sha256(lote / "b_copia.pdf")
    assert calcular_sha256(lote / "a_balance.pdf") != calcular_sha256(lote / "d_voucher.png")


def test_formato_por_firma_y_errores_tipificados(tmp_path):
    lote = _lote(tmp_path)
    assert detectar_formato(lote / "a_balance.pdf") == "pdf" and detectar_formato(lote / "c_vacio.pdf") == "vacio"
    with pytest.raises(ErrorArchivoVacio):
        validar_formato_y_extension(lote / "c_vacio.pdf", "vacio")
    falso = lote / "e_falso.pdf"
    falso.write_bytes((lote / "d_voucher.png").read_bytes())
    with pytest.raises(ErrorExtensionFalsa):
        validar_formato_y_extension(falso, detectar_formato(falso))


def test_pdf_cifrado(tmp_path):
    cifrado = _pdf(tmp_path / "c.pdf", encryption=pymupdf.PDF_ENCRYPT_AES_256, user_pw="x", owner_pw="y")
    with pytest.raises(ErrorPdfCifrado):
        verificar_apertura(cifrado, "pdf")


def test_cuarentena_copia_y_es_idempotente(tmp_path):
    origen = _pdf(tmp_path / "a.pdf")
    for _ in range(2):
        destino = enviar_a_cuarentena(origen, tmp_path / "q", "prueba")
    assert origen.exists() and destino.read_bytes() == origen.read_bytes()


def test_lote_un_registro_por_archivo_y_segunda_corrida_igual(tmp_path):
    cargador = CargadorDocumentos(_lote(tmp_path), tmp_path / "q")
    primera = cargador.cargar_lote()
    assert [r.estado for r in primera] == ["ok", "duplicado", "fallo", "ok"]
    assert primera[3].requiere_ocr and primera[2].motivo == "archivo_vacio"
    segunda = cargador.cargar_lote()
    assert [(r.archivo, r.estado, r.motivo) for r in primera] == [(r.archivo, r.estado, r.motivo) for r in segunda]
'''

raiz_pipeline = localizar_paquete()
if raiz_pipeline is None:
    print("No se encuentra el paquete clonado ni el respaldo; se usa el paquete ya instalado: la clase no se exporta.")
else:
    destino = raiz_pipeline / "pipeline_sbs" / "ingesta" / "cargador.py"
    actual = destino.read_text(encoding="utf-8")
    for objeto in (ResultadoCarga, CargadorDocumentos, tabla_resultados, estadisticas):
        marca = f"class {objeto.__name__}" if inspect.isclass(objeto) else f"def {objeto.__name__}("
        if marca in actual:
            print(f"{objeto.__name__} ya está en ingesta/cargador.py; no se duplica. Si se corrigió, se borra su versión anterior del módulo y se exporta de nuevo.")
            continue
        actual = actual.rstrip("\n") + "\n\n\n" + codigo_fuente(objeto)
        print(f"✔ {objeto.__name__} exportada a ingesta/cargador.py")
    destino.write_text(actual.rstrip("\n") + "\n", encoding="utf-8")
    pruebas = raiz_pipeline / "pipeline_sbs" / "tests" / "test_cargador.py"
    if pruebas.exists():
        print("tests/test_cargador.py ya existe; no se sobrescribe.")
    else:
        pruebas.parent.mkdir(parents=True, exist_ok=True)
        pruebas.write_text(PRUEBAS_CARGADOR, encoding="utf-8")
        print("✔ Pruebas mínimas agregadas a tests/test_cargador.py")

    import importlib
    import subprocess

    importlib.invalidate_caches()
    modulo_cargador = importlib.reload(importlib.import_module("pipeline_sbs.ingesta.cargador"))
    resultados_modulo = modulo_cargador.CargadorDocumentos(LOTE, Path(tempfile.mkdtemp(prefix="cuarentena_modulo_"))).cargar_lote()
    clave = ["archivo", "estado", "motivo", "formato", "paginas", "caracteres", "requiere_ocr", "sha256", "duplicado_de"]
    assert tabla_resultados(resultados)[clave].equals(modulo_cargador.tabla_resultados(resultados_modulo)[clave]), \
        "La clase exportada debe dar el mismo resultado que la del cuaderno: se ejecuta de nuevo la celda."
    corrida = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", str(Path(pipeline_sbs.__file__).parent / "tests")],
        capture_output=True, text=True, env=entorno_pruebas(), timeout=300,
    )
    print(corrida.stdout[-300:])
    assert corrida.returncode == 0, "pytest falla: el mensaje de arriba detalla el error."
    print("✔ Exportación superada: CargadorDocumentos está en pipeline_sbs.ingesta.cargador y las pruebas pasan")

### Guardar (≈1 min)

El participante pulsa `Ctrl+S` y ejecuta la celda de abajo: crea el commit en la VM y lo sube a `mio`, su repositorio personal (nunca a `origin`). **Debe verse:** `✔ Commit creado`, `✔ Subido a mio/trabajo-s02` y el último commit arriba de la lista. En github.com, el repositorio muestra el commit nuevo con ese mensaje.

In [ ]:
# Guardar el ejercicio: commit y subida a `mio`
guardar_avance("S02: E5 clase de carga")

## Caso — paso 1 (≈12 a 15 de los 50 min del caso) — Integra todo · M2.1 a M2.5: El primer pipeline básico, con un commit por etapa

**Contexto:** el equipo de la Mesa de Partes de *Financiera Altiplano DEMO S.A.* ya tiene la clase `CargadorDocumentos` (E5), que procesa un archivo a la vez. Ahora necesita una sola llamada, `ejecutar_pipeline(carpeta)`, que separe el trabajo en etapas (leer, normalizar, registrar errores y calcular estadísticas), cada una con una sola responsabilidad y su propio commit, y que devuelva una tabla con un registro por archivo, aunque algunos vengan dañados.

**Insumo:** 6 archivos de `02_Datos_Practica/01_Lote_S02/` (LOT-0001, LOT-0002, LOT-0007, LOT-0012, LOT-0016 y LOT-0019), que el checkpoint copia a una carpeta temporal. La clase de E5 sirve de referencia: el pipeline debe dar el mismo estado, motivo, caracteres y `requiere_ocr` que `CargadorDocumentos` en esos 6 archivos.

**Tarea:** partiendo de la clase de E5, el equipo reparte su lógica en etapas. El participante completa las etapas 2 a 4 (`normalizar_lote`, `registrar_errores` y `calcular_estadisticas`; la etapa 1, `leer_lote`, viene escrita y repite los pasos de `cargar_uno` hasta extraer) y encadena todo en `ejecutar_pipeline`. Cada celda de etapa termina exportando el módulo `orquestacion/pipeline.py` al paquete clonado; después de cada etapa, en PowerShell, dentro de la carpeta `entorno-extraccion-documental`, el participante ejecuta la celda «Guardar la etapa» que la sigue: registra un commit en la VM, sin subirlo (la subida se hace al cierre, con todas las etapas juntas).

**En palabras simples:** un pipeline es una línea de ensamblaje: el archivo entra, pasa por estaciones (leer, normalizar, registrar errores, estadísticas) y sale una fila de resultados por archivo. Un commit por etapa es como firmar el cuadro de avance cada vez que una estación queda lista. **Debe verse:** `✔ Checkpoint C6b superado` con 6 registros (2 ok, 3 fallos, 1 duplicado), igual que `CargadorDocumentos`, una segunda corrida idéntica y el módulo exportado funcionando igual. **Si falla:** un `AssertionError` indica qué etapa revisar; una etapa sin completar deja valores vacíos y el mensaje lo señala. Si `git commit` dice `nothing to commit`, la celda de la etapa no se ejecutó o ya se había registrado.

In [ ]:
# Piezas del caso (no se modifica): importaciones del paquete, parámetros y etapa 1 (leer el lote)
import logging
from typing import Any

from pipeline_sbs.config.configuracion import cargar_configuracion
from pipeline_sbs.ingesta.cargador import (
    TIPO_POR_EXTENSION,
    calcular_sha256,
    detectar_formato,
    enviar_a_cuarentena,
    validar_formato_y_extension,
    verificar_apertura,
)
from pipeline_sbs.ingesta.errores import ErrorIngesta, traducir_excepcion
from pipeline_sbs.ingesta.extractores import extraer_texto
from pipeline_sbs.preprocesamiento.normalizacion import normalizar_texto

registro = logging.getLogger(__name__)

FORMATOS_IMAGEN = ("png", "jpg", "tif")
COLUMNAS = ["archivo", "tipo", "formato", "estado", "motivo", "detalle", "paginas", "caracteres", "segundos",
            "requiere_ocr", "sha256", "duplicado_de"]


def _parametros(config: dict[str, Any] | None) -> dict[str, Any]:
    """Une la configuración recibida con la sección ``ingesta`` de ``pipeline.yaml``.

    Args:
        config: valores propios (``cuarentena``, ``umbral_ocr``) que reemplazan a los del YAML.

    Returns:
        Diccionario con ``cuarentena`` y ``umbral_ocr``.
    """
    ingesta = cargar_configuracion()["ingesta"]
    parametros = {
        "cuarentena": ingesta.get("carpeta_cuarentena", "cuarentena"),
        "umbral_ocr": ingesta.get("umbral_ocr_caracteres_por_pagina", 20),
    }
    parametros.update(config or {})
    return parametros


def leer_lote(carpeta: str | Path) -> list[dict[str, Any]]:
    """Etapa 1: abre cada archivo de la carpeta y extrae su texto sin detener el lote.

    Calcula la huella SHA-256, detecta el formato por la firma, valida contra la extensión, comprueba
    que el archivo abre y detecta duplicados por huella. El texto queda sin normalizar (``texto_crudo``).

    Args:
        carpeta: carpeta con los archivos recibidos (sin subcarpetas), leída en orden alfabético.

    Returns:
        Un diccionario por archivo con las columnas de ``COLUMNAS`` y ``texto_crudo``; ``ruta`` conserva
        el archivo original para la cuarentena.

    Raises:
        NotADirectoryError: si la carpeta no existe.
    """
    carpeta = Path(carpeta)
    if not carpeta.is_dir():
        raise NotADirectoryError(f"No existe la carpeta: {carpeta}")
    archivos = sorted(p for p in carpeta.iterdir() if p.is_file() and not p.name.startswith("."))
    vistos: dict[str, str] = {}        # huella -> nombre del primer archivo con ese contenido
    registros: list[dict[str, Any]] = []
    for ruta in archivos:
        inicio = time.perf_counter()
        fila: dict[str, Any] = {
            "archivo": ruta.name, "tipo": TIPO_POR_EXTENSION.get(ruta.suffix.lower(), "otro"), "formato": "desconocido",
            "estado": "ok", "motivo": "", "detalle": "", "paginas": 0, "caracteres": 0, "segundos": 0.0,
            "requiere_ocr": False, "sha256": "", "duplicado_de": "", "texto_crudo": "", "ruta": ruta,
        }
        try:
            fila["sha256"] = calcular_sha256(ruta)
            fila["formato"] = detectar_formato(ruta)
            validar_formato_y_extension(ruta, fila["formato"])
            verificar_apertura(ruta, fila["formato"])
            if fila["sha256"] in vistos:
                fila.update(estado="duplicado", motivo="duplicado", duplicado_de=vistos[fila["sha256"]],
                            detalle=f"mismo contenido que {vistos[fila['sha256']]}")
            else:
                extraccion = extraer_texto(ruta, fila["formato"])
                fila.update(texto_crudo=extraccion.texto, paginas=extraccion.paginas)
                vistos[fila["sha256"]] = ruta.name
        except Exception as error:  # noqa: BLE001 - un archivo malo no debe detener el lote
            tipificado = error if isinstance(error, ErrorIngesta) else traducir_excepcion(error)
            detalle = (tipificado.detalle or str(tipificado)).replace(str(ruta), ruta.name)
            fila.update(estado="fallo", motivo=tipificado.motivo, detalle=detalle)
        fila["segundos"] = time.perf_counter() - inicio
        registros.append(fila)
    return registros


In [ ]:
# Exportación del módulo orquestacion/pipeline.py (no se modifica)
ENCABEZADO_PIPELINE = '''"""Primer pipeline de punta a punta (S02): leer, normalizar, registrar errores y calcular estadísticas.

Cada etapa es una función con una sola responsabilidad. ``ejecutar_pipeline`` las encadena y devuelve
una tabla con un registro por archivo. Un archivo defectuoso nunca detiene el lote y repetir la corrida
sobre la misma carpeta produce el mismo resultado.
"""
from __future__ import annotations

import json
import logging
import time
from pathlib import Path
from typing import Any

import pandas as pd

from pipeline_sbs.config.configuracion import cargar_configuracion
from pipeline_sbs.ingesta.cargador import (
    TIPO_POR_EXTENSION,
    calcular_sha256,
    detectar_formato,
    enviar_a_cuarentena,
    validar_formato_y_extension,
    verificar_apertura,
)
from pipeline_sbs.ingesta.errores import ErrorIngesta, traducir_excepcion
from pipeline_sbs.ingesta.extractores import extraer_texto
from pipeline_sbs.preprocesamiento.normalizacion import normalizar_texto

registro = logging.getLogger(__name__)

FORMATOS_IMAGEN = ("png", "jpg", "tif")
COLUMNAS = ["archivo", "tipo", "formato", "estado", "motivo", "detalle", "paginas", "caracteres", "segundos",
            "requiere_ocr", "sha256", "duplicado_de"]
'''


def exportar_pipeline():
    """Escribe orquestacion/pipeline.py con las etapas definidas hasta ahora; se puede repetir (reemplaza el archivo)."""
    raiz = localizar_paquete()
    if raiz is None:
        print("Se usa el paquete ya instalado: no se escribe orquestacion/pipeline.py.")
        return
    nombres = ("_parametros", "leer_lote", "normalizar_lote", "registrar_errores", "calcular_estadisticas", "ejecutar_pipeline")
    etapas = [globals()[nombre] for nombre in nombres if nombre in globals()]
    destino = raiz / "pipeline_sbs" / "orquestacion" / "pipeline.py"
    destino.parent.mkdir(parents=True, exist_ok=True)
    cuerpo = "\n\n\n".join(codigo_fuente(etapa).rstrip("\n") for etapa in etapas)
    destino.write_text(ENCABEZADO_PIPELINE.rstrip("\n") + "\n\n\n" + cuerpo + "\n", encoding="utf-8")
    print("✔ Exportadas a orquestacion/pipeline.py:", ", ".join(etapa.__name__ for etapa in etapas))


exportar_pipeline()      # commit sugerido: "Etapa 1: leer el lote"


In [ ]:
# Guardar la etapa: commit en la VM (sin subir todavía)
guardar_avance("Etapa 1: leer el lote", subir=False)

In [ ]:
# Etapa del caso
def normalizar_lote(registros: list[dict[str, Any]], umbral_ocr: int = 20) -> list[dict[str, Any]]:
    """Etapa 2: normaliza el texto de los archivos ok y decide cuáles requieren OCR.

    Un archivo ok requiere OCR si es una imagen o un PDF con menos de ``umbral_ocr`` caracteres por página.

    Args:
        registros: salida de ``leer_lote``.
        umbral_ocr: caracteres por página por debajo de los cuales un PDF requiere OCR.

    Returns:
        Los mismos registros, con ``texto`` normalizado, ``caracteres`` y ``requiere_ocr`` completos.
    """
    for fila in registros:
        fila["texto"] = ""
        if fila["estado"] != "ok":
            continue
        texto = normalizar_texto(fila["texto_crudo"])
        por_pagina = len(texto) / max(fila["paginas"], 1)
        fila["texto"] = texto
        fila["caracteres"] = len(texto)
        fila["requiere_ocr"] = fila["formato"] in FORMATOS_IMAGEN or (fila["formato"] == "pdf" and por_pagina < umbral_ocr)
    return registros


exportar_pipeline()      # commit sugerido: "Etapa 2: normalizar"


In [ ]:
# Guardar la etapa: commit en la VM (sin subir todavía)
guardar_avance("Etapa 2: normalizar", subir=False)

In [ ]:
# Etapa del caso
def registrar_errores(registros: list[dict[str, Any]], cuarentena: str | Path) -> int:
    """Etapa 3: copia los archivos con fallo a la cuarentena y deja un evento por cada uno.

    Escribe además ``registro_errores.jsonl`` (una línea JSON por fallo) dentro de la cuarentena. Es
    idempotente: repetir la etapa reemplaza las copias y el registro.

    Args:
        registros: salida de ``normalizar_lote``.
        cuarentena: carpeta de cuarentena (se crea si hace falta).

    Returns:
        Cantidad de archivos con fallo.
    """
    cuarentena = Path(cuarentena)
    fallos = [fila for fila in registros if fila["estado"] == "fallo"]
    lineas = []
    for fila in fallos:
        try:
            enviar_a_cuarentena(fila["ruta"], cuarentena, fila["motivo"], fila["detalle"])
        except OSError as error:
            registro.error("cuarentena_fallida", extra={"archivo": fila["archivo"], "detalle": str(error)})
        registro.warning("archivo_con_error", extra={"archivo": fila["archivo"], "motivo": fila["motivo"]})
        lineas.append(json.dumps({"archivo": fila["archivo"], "motivo": fila["motivo"], "detalle": fila["detalle"]},
                                 ensure_ascii=False))
    if fallos:
        (cuarentena / "registro_errores.jsonl").write_text("\n".join(lineas) + "\n", encoding="utf-8")
    return len(fallos)


exportar_pipeline()      # commit sugerido: "Etapa 3: registrar errores"


In [ ]:
# Guardar la etapa: commit en la VM (sin subir todavía)
guardar_avance("Etapa 3: registrar errores", subir=False)

In [ ]:
# Etapa del caso
def calcular_estadisticas(tabla: pd.DataFrame) -> pd.DataFrame:
    """Etapa 4: resume el lote por tipo de archivo, con una fila final ``TOTAL``.

    Columnas: n, ok, fallo, duplicado, requiere_ocr y pct_requiere_ocr (solo sobre los archivos ok).

    Args:
        tabla: tabla con un registro por archivo (columnas ``tipo``, ``estado`` y ``requiere_ocr``).

    Returns:
        DataFrame indexado por tipo.
    """
    def resumir(grupo: pd.DataFrame) -> dict[str, Any]:
        ok = int((grupo["estado"] == "ok").sum())
        con_ocr = int(((grupo["estado"] == "ok") & grupo["requiere_ocr"]).sum())
        return {"n": len(grupo), "ok": ok, "fallo": int((grupo["estado"] == "fallo").sum()),
                "duplicado": int((grupo["estado"] == "duplicado").sum()), "requiere_ocr": con_ocr,
                "pct_requiere_ocr": round(100 * con_ocr / ok, 1) if ok else 0.0}

    columnas = ["n", "ok", "fallo", "duplicado", "requiere_ocr", "pct_requiere_ocr"]
    if tabla.empty:
        return pd.DataFrame(columns=columnas)
    filas = {tipo: resumir(grupo) for tipo, grupo in tabla.groupby("tipo", sort=True)}
    filas["TOTAL"] = resumir(tabla)
    return pd.DataFrame.from_dict(filas, orient="index")[columnas]


exportar_pipeline()      # commit sugerido: "Etapa 4: estadisticas"


In [ ]:
# Guardar la etapa: commit en la VM (sin subir todavía)
guardar_avance("Etapa 4: estadisticas", subir=False)

In [ ]:
# Etapa del caso
def ejecutar_pipeline(carpeta: str | Path, config: dict[str, Any] | None = None) -> pd.DataFrame:
    """Encadena leer, normalizar, registrar errores y estadísticas sobre una carpeta de archivos.

    Args:
        carpeta: carpeta con el lote (sin subcarpetas).
        config: valores que reemplazan a ``ingesta`` de ``pipeline.yaml``: ``cuarentena`` (carpeta de
            cuarentena) y ``umbral_ocr`` (caracteres por página).

    Returns:
        DataFrame con un registro por archivo (columnas de ``COLUMNAS``). Las estadísticas por tipo
        quedan en ``tabla.attrs["estadisticas"]``.

    Raises:
        NotADirectoryError: si la carpeta no existe.
    """
    parametros = _parametros(config)
    registros = leer_lote(carpeta)
    registros = normalizar_lote(registros, parametros["umbral_ocr"])
    errores = registrar_errores(registros, parametros["cuarentena"])
    tabla = pd.DataFrame([{columna: fila[columna] for columna in COLUMNAS} for fila in registros], columns=COLUMNAS)
    tabla.attrs["estadisticas"] = calcular_estadisticas(tabla)
    registro.info("pipeline_terminado", extra={"archivos": len(tabla), "errores": errores})
    return tabla


exportar_pipeline()      # commit sugerido: "Etapa 5: ejecutar_pipeline"


In [ ]:
# Guardar la etapa: commit en la VM (sin subir todavía)
guardar_avance("Etapa 5: ejecutar_pipeline", subir=False)

In [ ]:
# ✔ Checkpoint C6b: ejecutar_pipeline sobre 6 archivos del lote, comparado con la clase de E5
import importlib

SEIS = ("LOT-0001.pdf", "LOT-0002.pdf", "LOT-0007.pdf", "LOT-0012.pdf", "LOT-0016.pdf", "LOT-0019.pdf")
LOTE_CASO = Path(tempfile.mkdtemp(prefix="lote_caso_"))
for nombre in SEIS:
    shutil.copy2(LOTE / nombre, LOTE_CASO / nombre)

referencia_clase = {r.archivo: r for r in CargadorDocumentos(LOTE_CASO, Path(tempfile.mkdtemp(prefix="cuarentena_ref_"))).cargar_lote()}


def comprobar(funcion, etiqueta):
    cuarentena = Path(tempfile.mkdtemp(prefix="cuarentena_caso_"))
    tabla = funcion(LOTE_CASO, {"cuarentena": cuarentena})
    assert isinstance(tabla, pd.DataFrame), f"{etiqueta}: ejecutar_pipeline debe devolver un DataFrame."
    assert tabla["archivo"].tolist() == sorted(SEIS), f"{etiqueta}: debe haber un registro por archivo: {tabla['archivo'].tolist()}"
    por_archivo = tabla.set_index("archivo")
    esperado = {"LOT-0002.pdf": "extension_falsa", "LOT-0007.pdf": "pdf_cifrado", "LOT-0016.pdf": "archivo_vacio"}
    assert {a: por_archivo.loc[a, "motivo"] for a in esperado} == esperado, f"{etiqueta}: motivos {por_archivo['motivo'].to_dict()}"
    assert por_archivo.loc["LOT-0019.pdf", "estado"] == "duplicado" and por_archivo.loc["LOT-0019.pdf", "duplicado_de"] == "LOT-0001.pdf", \
        f"{etiqueta}: LOT-0019 es copia de LOT-0001."
    assert por_archivo.loc["LOT-0001.pdf", "caracteres"] > 1000 and bool(por_archivo.loc["LOT-0012.pdf", "requiere_ocr"]), \
        f"{etiqueta}: falta normalizar el texto y marcar requiere_ocr (etapa 2)."
    for archivo, esperado_clase in referencia_clase.items():
        fila = por_archivo.loc[archivo]
        assert (fila["estado"], fila["motivo"], int(fila["caracteres"]), bool(fila["requiere_ocr"])) == \
            (esperado_clase.estado, esperado_clase.motivo, esperado_clase.caracteres, esperado_clase.requiere_ocr), \
            f"{etiqueta}: {archivo} debe dar lo mismo que CargadorDocumentos (E5)."
    copias = sorted(p.name for p in cuarentena.iterdir() if not p.name.endswith((".json", ".jsonl")))
    assert copias == sorted(esperado), f"{etiqueta}: en cuarentena hay {copias} (etapa 3)."
    total = tabla.attrs.get("estadisticas")
    assert total is not None and total.loc["TOTAL", ["ok", "fallo", "duplicado"]].tolist() == [2, 3, 1], \
        f"{etiqueta}: faltan las estadísticas en tabla.attrs['estadisticas'] (etapa 4)."
    assert total.loc["TOTAL", "pct_requiere_ocr"] == 50.0, f"{etiqueta}: pct_requiere_ocr se calcula sobre los archivos ok (1 de 2 = 50,0)."
    segunda = funcion(LOTE_CASO, {"cuarentena": cuarentena})
    clave = ["archivo", "estado", "motivo", "formato", "paginas", "caracteres", "requiere_ocr", "sha256", "duplicado_de"]
    assert tabla[clave].equals(segunda[clave]), f"{etiqueta}: la segunda corrida debe ser idéntica."
    return tabla


tabla_caso = comprobar(ejecutar_pipeline, "cuaderno")
print(tabla_caso[["archivo", "estado", "motivo", "caracteres", "requiere_ocr"]].to_string(index=False))
print(tabla_caso.attrs["estadisticas"].to_string())

importlib.invalidate_caches()
nombre_modulo = "pipeline_sbs.orquestacion.pipeline"
modulo_pipeline = importlib.reload(importlib.import_module(nombre_modulo)) if nombre_modulo in sys.modules else importlib.import_module(nombre_modulo)
comprobar(modulo_pipeline.ejecutar_pipeline, "módulo exportado")
print("✔ Checkpoint C6b superado: 6 registros (2 ok, 3 fallos, 1 duplicado) iguales a los de CargadorDocumentos, segunda corrida idéntica y módulo exportado equivalente")

if RAIZ_PAQUETE is not None:
    propios = ejecutar_git("rev-list", "--count", "s02-inicio..HEAD")
    if propios is not None and propios.returncode == 0:
        print(f"Commits propios sobre s02-inicio: {propios.stdout.strip()} (el primer cambio, un Guardar por ejercicio y un commit por etapa).")

## Cierre del cuaderno — Verificación del objetivo: ¿qué leyó, qué limpió y cuántos documentos entraron?

El objetivo del módulo es implementar procesos de **ingesta, lectura, limpieza y normalización** de documentos de diferentes formatos. Esta celda lee la tabla de estadísticas del lote de 20 archivos (`resumen`, de E5) y responde las tres preguntas con cifras. En palabras simples: es el informe del día de la Mesa de Partes, con cuántos archivos se leyeron, qué se apartó y cuántos entraron. **Debe verse:** tres líneas, «Qué leyó», «Qué limpió» y «Cuántos entraron».

In [ ]:
# Las tres cifras del objetivo del módulo, desde la tabla de estadísticas del lote de 20 archivos
if "resumen" not in globals() or resumen is None:
    print("⚠ Falta la tabla `resumen`: se completa y se ejecuta E5.")
else:
    total = resumen.loc["TOTAL"]
    n, leidos = int(total["n"]), int(total["ok"])
    caracteres = f"{int(total['caracteres']):,}".replace(",", " ")
    print(f"Qué leyó: {leidos} de {n} archivos (PDF, DOCX, correo, HTML e imagen), con {caracteres} caracteres de texto; {int(total['requiere_ocr'])} requieren OCR.")
    print(f"Qué limpió: el texto de los {leidos} archivos leídos (NFC, espacios y guiones); {int(total['fallo'])} archivos defectuosos fueron a cuarentena y {int(total['duplicado'])} duplicado no se reprocesó.")
    print(f"Cuántos entraron: {leidos} de {n} ({100 * leidos / n:.1f} %) quedaron listos para el resto del pipeline.")

### Guardar el avance y la etiqueta de la sesión

Al terminar el caso, el participante pulsa `Ctrl+S` y ejecuta la celda de abajo: registra el avance con un commit, le pone la etiqueta `avance-s02` y lo sube todo a su repositorio personal (el cuaderno viaja con el commit). **Debe verse:** `✔ Commit creado`, `✔ Etiqueta avance-s02` y `✔ Subido a mio/trabajo-s02`. En github.com, la rama `trabajo-s02` muestra la etiqueta.

In [ ]:
# Guardar el avance de la sesión: commit, etiqueta y subida a `mio`
guardar_avance("Avance S02: pipeline del equipo", etiqueta="avance-s02")

### Checkpoint C7 (cierre): el cuaderno guardado en GitHub

Esta celda comprueba que el cuaderno `05_Laboratorios/01_LAB_S02_Ingesta.ipynb` está registrado en el repositorio clonado y sin cambios pendientes. En palabras simples: confirma que la última versión del cuaderno ya quedó firmada en un commit. La confirmación final se hace en el navegador: en github.com, el repositorio personal `extraccion-documental-sbs` muestra el cuaderno `05_Laboratorios/01_LAB_S02_Ingesta.ipynb` con sus resultados. **Debe verse:** `✔ C7: el cuaderno está en el repositorio y sin cambios pendientes` y, en github.com, la rama `trabajo-s02` con el cuaderno. **Si falla:** el aviso `cambios sin commit` significa que falta `Ctrl+S` o ejecutar la celda de «Guardar el avance y la etiqueta de la sesión» (se vuelve a ejecutar); si Git no está disponible o la carpeta no es un clon, la celda avisa y continúa: no bloquea el LAB.

In [ ]:
# ✔ Checkpoint C7 (cierre)
if RAIZ_PAQUETE is None:
    print("⚠ C7 omitido: el cuaderno no se abrió desde un repositorio clonado. Se avisa al docente.")
else:
    archivos = ejecutar_git("ls-files", "--", ":/05_Laboratorios")
    pendientes = ejecutar_git("status", "--porcelain", "--", ":/05_Laboratorios")
    if archivos is None or archivos.returncode != 0:
        print("⚠ C7 omitido: no se pudo leer `git ls-files` (Git no está en el PATH o la carpeta no es un clon).")
    elif "01_LAB_S02_Ingesta.ipynb" not in archivos.stdout:
        print("⚠ C7 pendiente: el cuaderno no está en el repositorio. Se abre desde la carpeta 05_Laboratorios del clon y se repite el Cierre.")
    elif pendientes is not None and pendientes.stdout.strip():
        print("⚠ C7 pendiente: hay cambios sin commit en 05_Laboratorios. Se pulsa Ctrl+S y se vuelve a ejecutar la celda «Guardar el avance y la etiqueta de la sesión».")
    else:
        print("✔ C7: el cuaderno está en el repositorio y sin cambios pendientes. Falta confirmar en github.com que se ve con sus resultados.")


## Reto extra (opcional)

El reto extra consiste en evaluar si 20 es un buen umbral de caracteres por página: el participante calcula los caracteres por página de cada PDF del lote que se abre sin error, propone un umbral que separe los escaneados de los nativos y analiza qué pasaría con un PDF que mezcla páginas con texto y páginas escaneadas.

In [ ]:
# Reto extra
por_pagina = {}
for ruta in sorted(LOTE.glob("*.pdf")):
    try:
        formato = detectar_formato(ruta)
        validar_formato_y_extension(ruta, formato)
        verificar_apertura(ruta, formato)
    except ErrorIngesta:
        continue
    extraccion = extraer_pdf_pymupdf(ruta)
    por_pagina[ruta.name] = extraccion.caracteres / extraccion.paginas
umbral_propuesto = 20
if umbral_propuesto is not None:
    escaneados = [v for v in por_pagina.values() if v < umbral_propuesto]
    nativos = [v for v in por_pagina.values() if v >= umbral_propuesto]
    assert escaneados and nativos, "El umbral debe separar al menos un escaneado de un nativo."
    assert max(escaneados) < umbral_propuesto <= min(nativos)
    print("Umbral razonable:", umbral_propuesto, "· caracteres por página:", {k: round(v) for k, v in por_pagina.items()})